In [ ]:
# ============================================================
# CELL 1 — Imports + config + paths
# ============================================================
# %pip install -q transformers sentencepiece

import os
import re
import ast
import math
import json
import time
import random
import pickle
from collections import Counter, defaultdict
from functools import lru_cache

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# Repro
# -------------------------
def seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

if device.type == "cpu":
    try:
        torch.set_num_threads(max(1, os.cpu_count() // 2))
    except Exception:
        pass

# -------------------------
# Exact file paths
# -------------------------
INTERACTIONS_CSV  = "/teamspace/studios/this_studio/SPR-NEURIPS_interactionn_ordered_summ_and_rec.csv"
NEWS_CSV          = "/teamspace/studios/this_studio/SPR-NEURIPS_pens_news (2).csv"
SUMMARIES_CSV     = "/teamspace/studios/this_studio/SPR-NEURIPS_summaries_ext.csv"
NEWS_BODY_EMB_PKL = "/teamspace/studios/this_studio/SPR-NEURIPS_newsbody_embeddings.pkl"
SUM_T5_EMB_PKL    = "/teamspace/studios/this_studio/SPR-NEURIPS_trainsum_T5.pkl"

EXT_MAIN_TEST_CSV = "/teamspace/studios/this_studio/SPR-NEURIPS_PerDucer_exp_Test.csv"
EXT_TESTSUM_CSV   = "/teamspace/studios/this_studio/testsumm_ext.csv"
EXT_TESTSUM_PKL   = "/teamspace/studios/this_studio/SPR-NEURIPS_testsum_T5.pkl"

ROOT_OUT = "/teamspace/studios/this_studio"
CKPT_DIR = os.path.join(ROOT_OUT, "ckpt_pertide_rec_sum_fixed")
EVAL_DIR = os.path.join(ROOT_OUT, "eval_outputs_pertide_rec_sum_fixed")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

# -------------------------
# Split protocol
# -------------------------
RAW_LIMIT = 75000
TRAIN_N = 63000
VAL_N   = 1000
TEST_N  = 11000
TOTAL_N = TRAIN_N + VAL_N + TEST_N
SPLIT_SEED = 32

# -------------------------
# Model/data limits
# -------------------------
MAX_HIS_LEN = 50
MAX_SRC_TOK = 192
MAX_TGT_TOK = 64
MAX_DOC_BODY_CHARS = 2400

# -------------------------
# Model dims / tasks
# -------------------------
D_MODEL = 768
TASK_NAMES = ["rec", "sum"]
TASK2ID = {t:i for i, t in enumerate(TASK_NAMES)}
ID2TASK = {i:t for t, i in TASK2ID.items()}

# -------------------------
# Training setup
# -------------------------
REC_BATCH = 12 if device.type == "cuda" else 16
SUM_BATCH = 12 if device.type == "cuda" else 4
VAL_SUM_BATCH = 8 if device.type == "cuda" else 2

STAGE1_REC_EPOCHS = 4
STAGE2_JOINT_EPOCHS = 4
STAGE3_SUM_FT_EPOCHS = 4

STAGE1_REC_LR = 2e-4
STAGE2_ENCODER_LR = 8e-5
STAGE2_CONTEXT_LR = 3e-4
STAGE2_DECODER_LR = 5e-5
STAGE3_CONTEXT_LR = 2.5e-4
STAGE3_DECODER_LR = 4e-5
WD = 1e-4

STAGE1_WARMUP_RATIO = 0.10
STAGE2_WARMUP_RATIO = 0.06
STAGE3_WARMUP_RATIO = 0.10

GRAD_CLIP = 1.0
NUM_WORKERS = 0
GRAD_ACCUM_STEPS = 1

REC_TRAIN_NEGS = 63
REC_EVAL_NEGS  = 499
ENC_NEXT_NEGS  = 1023

USE_AMP = (device.type == "cuda")
USE_COMPILE = False

# -------------------------
# Decoder / conditioning
# -------------------------
DECODER_NAME = "distilgpt2"
PREFIX_LEN = 8
USE_UGI = True
DECODER_TRAIN_LAST_N_BLOCKS = 2
TRAIN_DECODER_LN_F = True
SEM_LOSS_WEIGHT = 0.15
DROPOUT = 0.1

# -------------------------
# Encoder loss weights
# Lenc = 0.2 Lpos + 0.4 Lnext + 0.4 Ltask
# -------------------------
LAMBDA_POS  = 0.2
LAMBDA_NEXT = 0.4
LAMBDA_TASK = 0.4

LAMBDA_REC = 1.0
LAMBDA_SUM = 1.0

# -------------------------
# Action vocabulary
# -------------------------
ACTION2ID = {"PAD": 0, "pos": 1, "neu": 2, "neg": 3}
ID2ACTION = {v: k for k, v in ACTION2ID.items()}
NUM_ACTIONS = len(ACTION2ID)

# -------------------------
# Leakage thresholds
# -------------------------
MAX_ACCEPTABLE_LAST_EQ_RATE_TRAIN = 0.001
MAX_ACCEPTABLE_LAST_EQ_RATE_VAL   = 0.01
MAX_ACCEPTABLE_LAST_EQ_RATE_TEST  = 0.001

# -------------------------
# PerSEval requirement
# -------------------------
PSE_MIN_USERS_PER_DOC = 2

# -------------------------
# Output files
# -------------------------
STAGE1_LAST_CKPT = os.path.join(CKPT_DIR, "stage1_rec_last.pt")
STAGE1_BEST_CKPT = os.path.join(CKPT_DIR, "stage1_rec_best.pt")
STAGE2_LAST_CKPT = os.path.join(CKPT_DIR, "stage2_joint_last.pt")
STAGE2_BEST_CKPT = os.path.join(CKPT_DIR, "stage2_joint_best.pt")
STAGE3_LAST_CKPT = os.path.join(CKPT_DIR, "stage3_sumft_last.pt")
STAGE3_BEST_CKPT = os.path.join(CKPT_DIR, "stage3_sumft_best.pt")

STAGE1_HIST_CSV = os.path.join(EVAL_DIR, "stage1_history.csv")
STAGE2_HIST_CSV = os.path.join(EVAL_DIR, "stage2_history.csv")
STAGE3_HIST_CSV = os.path.join(EVAL_DIR, "stage3_history.csv")

print("config ready")

In [2]:
# ============================================================
# CELL 2 — Load raw internal + external files
# ============================================================
for p in [INTERACTIONS_CSV, NEWS_CSV, SUMMARIES_CSV, NEWS_BODY_EMB_PKL, SUM_T5_EMB_PKL]:
    if not os.path.exists(p):
        raise FileNotFoundError(p)

news_df = pd.read_csv(NEWS_CSV)
sum_df  = pd.read_csv(SUMMARIES_CSV)
int_df  = pd.read_csv(INTERACTIONS_CSV)

if RAW_LIMIT is not None:
    int_df = int_df.iloc[:RAW_LIMIT].copy()

with open(NEWS_BODY_EMB_PKL, "rb") as f:
    news_body_emb = pickle.load(f)

with open(SUM_T5_EMB_PKL, "rb") as f:
    sum_t5_emb = pickle.load(f)

assert isinstance(news_body_emb, dict), "news_body_emb must be dict"
assert isinstance(sum_t5_emb, dict), "sum_t5_emb must be dict"

print("INTERNAL")
print("news_df:", news_df.shape)
print("sum_df :", sum_df.shape)
print("int_df :", int_df.shape)
print("news_body_emb:", len(news_body_emb))
print("sum_t5_emb   :", len(sum_t5_emb))

ext_main_df = None
ext_sum_df = None
ext_sum_pkl = None

if all(os.path.exists(p) for p in [EXT_MAIN_TEST_CSV, EXT_TESTSUM_CSV, EXT_TESTSUM_PKL]):
    ext_main_df = pd.read_csv(EXT_MAIN_TEST_CSV).copy()
    ext_sum_df  = pd.read_csv(EXT_TESTSUM_CSV).copy()
    with open(EXT_TESTSUM_PKL, "rb") as f:
        ext_sum_pkl = pickle.load(f)

    print("\nEXTERNAL")
    print("ext_main_df:", ext_main_df.shape)
    print("ext_sum_df :", ext_sum_df.shape)
    if isinstance(ext_sum_pkl, dict):
        print("ext_sum_pkl: dict", len(ext_sum_pkl))
    elif isinstance(ext_sum_pkl, (list, tuple)):
        print("ext_sum_pkl: seq", len(ext_sum_pkl))
    else:
        print("ext_sum_pkl type:", type(ext_sum_pkl))
else:
    print("\nExternal files not found; external eval will be skipped until files are present.")

In [3]:
# ============================================================
# CELL 3 — Normalize schema + build internal text / summary lookup
# ============================================================
NEWSID_RE = re.compile(r"\bN\d+\b", re.IGNORECASE)
SENT_RE   = re.compile(r"\b(pos|neu|neg)\b", re.IGNORECASE)

def normalize_news_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    m = NEWSID_RE.search(s)
    return m.group(0).upper() if m else s

def normalize_action(x):
    s = str(x).strip().lower()
    if s in {"pos", "positive", "click", "clicked", "open", "opened", "read", "liked", "like"}:
        return "pos"
    if s in {"neg", "negative", "skip", "skipped", "hide", "hidden", "dislike", "disliked"}:
        return "neg"
    return "neu"

def normalize_sid(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s.startswith("SUMM"):
        return s
    if s.startswith("S") and s[1:].isdigit():
        return "SUMM" + s[1:]
    m = re.search(r"(\d+)", s)
    return ("SUMM" + m.group(1)) if m else s

def sid_to_num(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d+)", str(x))
    return int(m.group(1)) if m else np.nan

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\n", " ").strip()

def parse_history_cell(x):
    if isinstance(x, list):
        out = []
        for it in x:
            if isinstance(it, (list, tuple)) and len(it) >= 2:
                out.append((normalize_news_id(it[0]), normalize_action(it[1])))
        return [(a, b) for a, b in out if a is not None]

    s = str(x)
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            out = []
            for t in obj:
                if isinstance(t, (list, tuple)) and len(t) >= 2:
                    out.append((normalize_news_id(t[0]), normalize_action(t[1])))
            out = [(a, b) for a, b in out if a is not None]
            if len(out) > 0:
                return out
    except Exception:
        pass

    mids = [m.upper() for m in NEWSID_RE.findall(s)]
    acts = [normalize_action(z) for z in SENT_RE.findall(s)]
    if len(acts) < len(mids):
        acts = acts + ["neu"] * (len(mids) - len(acts))
    return list(zip(mids, acts))

# -------------------------
# Internal interaction table
# -------------------------
need_cols = {"userid", "history", "query_id", "target_id"}
if not need_cols.issubset(set(int_df.columns)):
    raise KeyError(f"Need columns {need_cols}, found {set(int_df.columns)}")

int_df["userid"]    = int_df["userid"].astype(str)
int_df["query_id"]  = int_df["query_id"].apply(normalize_news_id)
int_df["target_id"] = int_df["target_id"].apply(normalize_sid)

tqdm.pandas(desc="parsing internal interaction history")
int_df["history_parsed"] = int_df["history"].progress_apply(parse_history_cell)

# -------------------------
# Internal summary table + STRICT embedding alignment
# -------------------------
sum_df["SummID"] = sum_df["SummID"].apply(normalize_sid)
sum_df["NewsID"] = sum_df["NewsID"].apply(normalize_news_id)
sum_df["UserID"] = sum_df["UserID"].astype(str)
sum_df["extended_summary"] = sum_df["extended_summary"].astype(str)

if not isinstance(sum_t5_emb, dict):
    raise TypeError(f"sum_t5_emb must be dict, got {type(sum_t5_emb)}")

def extract_internal_sum_emb(v):
    """
    Strictly extract the actual 1D embedding vector from the pickle value.
    No fallback zeros.
    """
    if v is None:
        return None

    if isinstance(v, dict):
        for kk in ["emb", "embedding", "sum_emb", "summary_emb", "t5_emb"]:
            if kk in v:
                arr = np.asarray(v[kk], dtype=np.float32)
                if arr.ndim == 1:
                    return arr
        return None

    arr = np.asarray(v, dtype=np.float32)
    if arr.ndim == 1:
        return arr
    return None

# normalize embedding keys into same SID space as dataframe
sum_t5_emb_norm = {}
bad_value_count = 0

for raw_k, raw_v in tqdm(sum_t5_emb.items(), total=len(sum_t5_emb), desc="normalizing internal summary embedding keys"):
    sid_norm = normalize_sid(raw_k)
    emb = extract_internal_sum_emb(raw_v)

    if sid_norm is None:
        continue
    if emb is None:
        bad_value_count += 1
        continue

    # keep first if duplicates appear after normalization
    if sid_norm not in sum_t5_emb_norm:
        sum_t5_emb_norm[sid_norm] = emb

if len(sum_t5_emb_norm) == 0:
    raise RuntimeError(
        "After SID normalization and embedding extraction, no usable summary embeddings were found in SUM_T5_EMB_PKL. "
        "This indicates the pickle format or key-space is different from the dataframe SummID space."
    )

sum_emb_dim = int(next(iter(sum_t5_emb_norm.values())).shape[0])

summary_lookup = {}
matched_count = 0
missing_ids = []

for sid, nid, uid, txt in tqdm(
    zip(
        sum_df["SummID"].tolist(),
        sum_df["NewsID"].tolist(),
        sum_df["UserID"].tolist(),
        sum_df["extended_summary"].tolist(),
    ),
    total=len(sum_df),
    desc="building internal summary lookup"
):
    emb = sum_t5_emb_norm.get(sid, None)
    if emb is None:
        missing_ids.append(sid)
        continue

    emb = np.asarray(emb, dtype=np.float32)
    if emb.ndim != 1:
        raise ValueError(f"Embedding for {sid} is not 1D: shape={emb.shape}")
    if emb.shape[0] != sum_emb_dim:
        raise ValueError(f"Embedding dim mismatch for {sid}: got {emb.shape[0]}, expected {sum_emb_dim}")

    matched_count += 1
    summary_lookup[sid] = {
        "newsid": nid,
        "userid": uid,
        "text": txt,
        "emb": emb,
    }

print("internal summary rows:", len(sum_df))
print("normalized embedding keys:", len(sum_t5_emb_norm))
print("bad embedding values skipped:", bad_value_count)
print("matched summary rows:", matched_count)
print("missing summary rows:", len(missing_ids))
print("sum_emb_dim:", sum_emb_dim)

if matched_count == 0:
    raise RuntimeError(
        "No internal summaries matched normalized embedding keys. "
        "Your SUM_T5_EMB_PKL key format does not align with normalized SummID values."
    )

# strict requirement: since you said to use embeddings properly, do not proceed with broad mismatch silently
match_rate = matched_count / max(1, len(sum_df))
print("embedding match rate:", round(match_rate, 6))

# inspect first few missing ids if mismatch remains
if len(missing_ids) > 0:
    print("example missing SummIDs:", missing_ids[:20])

# keep only rows whose true summary embedding exists
sum_df = sum_df[sum_df["SummID"].isin(summary_lookup.keys())].reset_index(drop=True)
print("sum_df after strict embedding alignment:", sum_df.shape)
# -------------------------
# Internal news table
# -------------------------
colmap = {c.lower(): c for c in news_df.columns}
def pick(*cands):
    for c in cands:
        if c.lower() in colmap:
            return colmap[c.lower()]
    raise KeyError(f"Missing any of {cands}. Available = {list(news_df.columns)}")

NEWSID_COL = pick("NewsID", "newsid")
HEAD_COL   = pick("Headline", "headline", "title")
BODY_COL   = pick("NewsBody", "body", "content", "abstract")
CAT_COL    = pick("Category", "category", "vert", "newsvert")
TOPIC_COL  = pick("Topic", "topic", "subvert", "newssubvert")

news_df[NEWSID_COL] = news_df[NEWSID_COL].apply(normalize_news_id)

news_text_lookup = {}
for _, r in tqdm(news_df.iterrows(), total=len(news_df), desc="building internal news text lookup"):
    nid = r[NEWSID_COL]
    head = safe_text(r[HEAD_COL])
    body = safe_text(r[BODY_COL])
    cat  = safe_text(r[CAT_COL])
    top  = safe_text(r[TOPIC_COL])

    source_text = f"category: {cat}. topic: {top}. headline: {head}. article: {body[:MAX_DOC_BODY_CHARS]}".strip()
    news_text_lookup[nid] = {
        "headline": head,
        "body": body,
        "category": cat,
        "topic": top,
        "source_text": source_text,
    }

# -------------------------
# Internal task subsets
# rec rows: any row with a valid query_id
# sum rows: any row with a valid normalized summary id target
# -------------------------
is_rec = int_df["query_id"].notna()
is_sum = int_df["target_id"].notna() & int_df["target_id"].str.startswith("SUMM", na=False)

rec_df = int_df[is_rec].copy().reset_index(drop=True)
sum_int_df = int_df[is_sum].copy().reset_index(drop=True)

print("rec_df    :", rec_df.shape)
print("sum_int_df:", sum_int_df.shape)

# optional summary-row dedup by true summary id
before_dedup = len(sum_int_df)
sum_int_df = sum_int_df.drop_duplicates(subset=["target_id"]).reset_index(drop=True)
print("sum_int_df after sid dedup:", sum_int_df.shape, "| removed:", before_dedup - len(sum_int_df))

normalizing internal summary embedding keys:   0%|          | 0/135001 [00:00<?, ?it/s]

building internal summary lookup:   0%|          | 0/135001 [00:00<?, ?it/s]

internal summary rows: 135001
normalized embedding keys: 135001
bad embedding values skipped: 0
matched summary rows: 135001
missing summary rows: 0
sum_emb_dim: 768
embedding match rate: 1.0
sum_df after strict embedding alignment: (135001, 4)


building internal news text lookup:   0%|          | 0/113762 [00:00<?, ?it/s]

rec_df    : (75000, 6)
sum_int_df: (75000, 6)
sum_int_df after sid dedup: (74931, 6) | removed: 69


In [ ]:
# ============================================================
# CELL 4 — Build ID maps + tokenize news docs + tokenize INTERNAL targets
# ============================================================
# -------------------------
# Build unified news id map from embeddings + news table + summaries + external summaries if available
# -------------------------
all_news_ids = set()

for k in news_body_emb.keys():
    nk = normalize_news_id(k)
    if nk is not None:
        all_news_ids.add(nk)

for nid in news_df[NEWSID_COL].dropna().tolist():
    nk = normalize_news_id(nid)
    if nk is not None:
        all_news_ids.add(nk)

for sid, info in summary_lookup.items():
    nk = normalize_news_id(info["newsid"])
    if nk is not None:
        all_news_ids.add(nk)

if ext_sum_df is not None:
    # use raw external summary file so tokenizer / news map knows every possible external target news id
    ext_sum_newsid_col = None
    for c in ["NewsID", "newsid"]:
        if c in ext_sum_df.columns:
            ext_sum_newsid_col = c
            break
    if ext_sum_newsid_col is not None:
        for nid in ext_sum_df[ext_sum_newsid_col].dropna().tolist():
            nk = normalize_news_id(nid)
            if nk is not None:
                all_news_ids.add(nk)

all_news_ids = sorted(all_news_ids)

news2id = {"<PAD>": 0, "<UNK>": 1}
for nid in all_news_ids:
    if nid not in news2id:
        news2id[nid] = len(news2id)

id2news = {v: k for k, v in news2id.items()}
num_news = len(news2id)

any_news_emb = next(iter(news_body_emb.values()))
raw_news_emb_dim = int(np.asarray(any_news_emb).shape[0])

news_emb_arr = np.zeros((num_news, raw_news_emb_dim), dtype=np.float32)
source_text_arr = [""] * num_news
headline_arr = [""] * num_news

for nid in tqdm(all_news_ids, desc="caching internal news embeddings/text"):
    idx = news2id[nid]

    emb = news_body_emb.get(nid, None)
    if emb is None:
        emb = np.zeros((raw_news_emb_dim,), dtype=np.float32)
    else:
        emb = np.asarray(emb, dtype=np.float32)
        if emb.ndim != 1 or emb.shape[0] != raw_news_emb_dim:
            raise ValueError(f"bad embedding for {nid}: {emb.shape}")

    news_emb_arr[idx] = emb

    meta = news_text_lookup.get(nid, None)
    if meta is None:
        source_text_arr[idx] = ""
        headline_arr[idx] = ""
    else:
        source_text_arr[idx] = meta["source_text"]
        headline_arr[idx] = meta["headline"]

# -------------------------
# Internal summary source lookup
# -------------------------
sum_src_nid = {}
sum_text_lookup = {}
sum_emb_lookup = {}
for sid, info in summary_lookup.items():
    sum_src_nid[sid] = news2id.get(info["newsid"], news2id["<UNK>"])
    sum_text_lookup[sid] = info["text"]
    if info["emb"] is None:
        sum_emb_lookup[sid] = np.zeros((sum_emb_dim,), dtype=np.float32)
    else:
        sum_emb_lookup[sid] = np.asarray(info["emb"], dtype=np.float32)

# -------------------------
# Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(DECODER_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tok_text(text, max_len):
    enc = tokenizer(
        str(text),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="np",
    )
    return enc["input_ids"][0].astype(np.int64), enc["attention_mask"][0].astype(np.int64)

# -------------------------
# Tokenize all source docs
# -------------------------
SRC_IDS_ARR  = np.zeros((num_news, MAX_SRC_TOK), dtype=np.int64)
SRC_MASK_ARR = np.zeros((num_news, MAX_SRC_TOK), dtype=np.int64)

for idx in tqdm(range(num_news), desc="tokenizing source docs"):
    ids, mask = tok_text(source_text_arr[idx], MAX_SRC_TOK)
    SRC_IDS_ARR[idx] = ids
    SRC_MASK_ARR[idx] = mask

# -------------------------
# Tokenize all INTERNAL gold summaries
# -------------------------
SUM_TARGET_CACHE = {}
for sid, txt in tqdm(sum_text_lookup.items(), total=len(sum_text_lookup), desc="tokenizing internal targets"):
    ids, mask = tok_text(txt, MAX_TGT_TOK)
    SUM_TARGET_CACHE[sid] = {
        "ids": ids,
        "mask": mask,
        "text": txt,
        "emb": sum_emb_lookup[sid],
        "src_nid": sum_src_nid[sid],
        "newsid": summary_lookup[sid]["newsid"],
        "userid": summary_lookup[sid]["userid"],
    }

# external target cache gets built in Cell 6 after corrected external join
EXT_TARGET_CACHE = {}

NEWS_EMB_T   = torch.tensor(news_emb_arr, dtype=torch.float32)
SRC_IDS_T    = torch.tensor(SRC_IDS_ARR, dtype=torch.long)
SRC_MASK_T   = torch.tensor(SRC_MASK_ARR, dtype=torch.long)

print("num_news:", num_news)
print("raw_news_emb_dim:", raw_news_emb_dim)
print("tokenizer vocab:", tokenizer.vocab_size)
print("internal target cache:", len(SUM_TARGET_CACHE))

caching internal news embeddings/text:   0%|          | 0/113762 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# CELL 5 — Leakage-safe INTERNAL encoding + exact train/val/test split
# IMPORTANT FIX:
#   For summarization, qid is the true source document to be summarized.
#   So qid = src_nid, not the interaction-side query_id.
# ============================================================
def strip_terminal_target_from_history(hist_parsed, target_newsid_norm):
    """
    Remove exactly one leaked terminal occurrence of the target document from history.
    We only remove the final leaked target occurrence.
    Earlier genuine repeats are preserved.
    """
    if hist_parsed is None:
        return []
    hist = list(hist_parsed)
    if target_newsid_norm is None or len(hist) == 0:
        return hist

    if normalize_news_id(hist[-1][0]) == target_newsid_norm:
        hist = hist[:-1]
    return hist

def encode_history(hist_parsed, max_len=MAX_HIS_LEN):
    hist = hist_parsed[-max_len:] if len(hist_parsed) > max_len else hist_parsed
    nids = [news2id.get(normalize_news_id(n), news2id["<UNK>"]) for (n, a) in hist]
    acts = [ACTION2ID.get(normalize_action(a), ACTION2ID["neu"]) for (n, a) in hist]
    L = len(nids)

    if L < max_len:
        nids += [0] * (max_len - L)
        acts += [0] * (max_len - L)

    return np.asarray(nids, dtype=np.int64), np.asarray(acts, dtype=np.int64), int(L)

def encode_rec_row(r):
    qid_norm = normalize_news_id(r["query_id"])
    hist_clean = strip_terminal_target_from_history(r["history_parsed"], qid_norm)
    h_ids, h_act, h_len = encode_history(hist_clean, MAX_HIS_LEN)
    qid = news2id.get(qid_norm, news2id["<UNK>"])
    return h_ids, h_act, h_len, qid

def encode_sum_row(r):
    sid = str(r["target_id"])
    if sid not in SUM_TARGET_CACHE:
        return None

    # true summary anchor = summary source document
    target_newsid_norm = normalize_news_id(SUM_TARGET_CACHE[sid]["newsid"])
    src_nid = int(SUM_TARGET_CACHE[sid]["src_nid"])
    qid = src_nid

    hist_clean = strip_terminal_target_from_history(r["history_parsed"], target_newsid_norm)
    h_ids, h_act, h_len = encode_history(hist_clean, MAX_HIS_LEN)

    return h_ids, h_act, h_len, src_nid, qid, sid, r["userid"]

# -------------------------
# Encode REC rows
# -------------------------
tqdm.pandas(desc="encoding REC rows (de-leaked)")
rec_rows = rec_df.progress_apply(encode_rec_row, axis=1)

rec_df = rec_df.copy().reset_index(drop=True)
rec_df["h_ids"] = [x[0] for x in rec_rows]
rec_df["h_act"] = [x[1] for x in rec_rows]
rec_df["h_len"] = [x[2] for x in rec_rows]
rec_df["qid"]   = [x[3] for x in rec_rows]

# -------------------------
# Encode SUM rows
# -------------------------
tqdm.pandas(desc="encoding SUM rows (de-leaked, qid=src_nid)")
sum_rows = sum_int_df.progress_apply(encode_sum_row, axis=1)
sum_valid_mask = sum_rows.notna()

sum_int_df = sum_int_df[sum_valid_mask].reset_index(drop=True)
sum_rows = sum_rows[sum_valid_mask].reset_index(drop=True)

sum_int_df["h_ids"]   = [x[0] for x in sum_rows]
sum_int_df["h_act"]   = [x[1] for x in sum_rows]
sum_int_df["h_len"]   = [x[2] for x in sum_rows]
sum_int_df["src_nid"] = [x[3] for x in sum_rows]
sum_int_df["qid"]     = [x[4] for x in sum_rows]
sum_int_df["sid"]     = [x[5] for x in sum_rows]
sum_int_df["userid"]  = [x[6] for x in sum_rows]

# -------------------------
# Split helpers
# REC stays exact: 65000 / 500 / 9500
# SUM uses all available rows after strict embedding filtering:
#   keep val=500, test=9500 fixed
#   reduce only train if total < 75000
# -------------------------
def exact_split(df, seed, name):
    """
    Exact split for tasks that still have at least TOTAL_N rows.
    Used for recommendation.
    """
    if len(df) < TOTAL_N:
        raise ValueError(f"{name}: need at least {TOTAL_N} rows, found {len(df)}")

    df_s = df.sample(n=TOTAL_N, random_state=seed).reset_index(drop=True)
    tr = df_s.iloc[:TRAIN_N].reset_index(drop=True)
    va = df_s.iloc[TRAIN_N:TRAIN_N + VAL_N].reset_index(drop=True)
    te = df_s.iloc[TRAIN_N + VAL_N:TRAIN_N + VAL_N + TEST_N].reset_index(drop=True)
    return tr, va, te


def capped_sum_split(df, seed, name):
    """
    Summarization split after strict embedding-alignment.
    Preserve:
      val  = 500
      test = 9500
    Reduce only train if total is short.
    """
    min_needed = VAL_N + TEST_N
    if len(df) < min_needed:
        raise ValueError(
            f"{name}: need at least {min_needed} rows to preserve val/test, found {len(df)}"
        )

    usable_n = min(len(df), TOTAL_N)
    df_s = df.sample(n=usable_n, random_state=seed).reset_index(drop=True)

    # keep val/test fixed
    train_n_eff = usable_n - VAL_N - TEST_N
    if train_n_eff <= 0:
        raise ValueError(
            f"{name}: effective train split became non-positive ({train_n_eff})."
        )

    tr = df_s.iloc[:train_n_eff].reset_index(drop=True)
    va = df_s.iloc[train_n_eff:train_n_eff + VAL_N].reset_index(drop=True)
    te = df_s.iloc[train_n_eff + VAL_N:train_n_eff + VAL_N + TEST_N].reset_index(drop=True)

    return tr, va, te, train_n_eff


# -------------------------
# Exact REC split
# -------------------------
rec_tr, rec_va, rec_te = exact_split(rec_df, SPLIT_SEED, "REC")

# -------------------------
# Capped SUM split
# remove shortage only from SUM-train
# -------------------------
sum_tr, sum_va, sum_te, SUM_TRAIN_N_EFF = capped_sum_split(sum_int_df, SPLIT_SEED + 1, "SUM")

print("rec split:", len(rec_tr), len(rec_va), len(rec_te))
print("sum split:", len(sum_tr), len(sum_va), len(sum_te))
print("effective summarization train size:", SUM_TRAIN_N_EFF)
print("summarization rows dropped from train due to missing embeddings:", TRAIN_N - SUM_TRAIN_N_EFF)

assert len(rec_tr) == TRAIN_N and len(rec_va) == VAL_N and len(rec_te) == TEST_N
assert len(sum_va) == VAL_N and len(sum_te) == TEST_N
assert len(sum_tr) == len(sum_int_df) - VAL_N - TEST_N


def leakage_stats(df, name):
    leak_last = 0
    in_hist = 0
    bad_len = 0

    for _, r in df.iterrows():
        h = r["h_ids"]
        L = int(r["h_len"])
        q = int(r["qid"])

        if L < 0 or L > len(h):
            bad_len += 1
            continue

        hist = list(h[:L]) if isinstance(h, np.ndarray) else list(h)[:L]

        if len(hist) > 0 and hist[-1] == q:
            leak_last += 1
        if q in hist:
            in_hist += 1

    stats = {
        "rows": int(len(df)),
        "last_item_equals_qid_count": int(leak_last),
        "last_item_equals_qid_rate": float(leak_last / max(1, len(df))),
        "label_in_history_count": int(in_hist),
        "label_in_history_rate": float(in_hist / max(1, len(df))),
        "bad_hlen_count": int(bad_len),
    }

    print(
        f"{name}: rows={stats['rows']} "
        f"last_item_equals_qid_rate={stats['last_item_equals_qid_rate']:.6f} "
        f"label_in_history_rate={stats['label_in_history_rate']:.6f}"
    )
    return stats

print("rec split:", len(rec_tr), len(rec_va), len(rec_te))
print("sum split:", len(sum_tr), len(sum_va), len(sum_te))

# recommendation leakage
rec_tr_stats = leakage_stats(rec_tr, "rec_tr")
rec_va_stats = leakage_stats(rec_va, "rec_va")
rec_te_stats = leakage_stats(rec_te, "rec_te")

# summarization leakage
sum_tr_stats = leakage_stats(sum_tr, "sum_tr")
sum_va_stats = leakage_stats(sum_va, "sum_va")
sum_te_stats = leakage_stats(sum_te, "sum_te")

# strict no-bad-length checks
for nm, st in [
    ("rec_tr", rec_tr_stats), ("rec_va", rec_va_stats), ("rec_te", rec_te_stats),
    ("sum_tr", sum_tr_stats), ("sum_va", sum_va_stats), ("sum_te", sum_te_stats),
]:
    assert st["bad_hlen_count"] == 0, f"{nm} has invalid h_len rows: {st['bad_hlen_count']}"

# bounded terminal leakage checks
assert rec_tr_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_TRAIN
assert rec_va_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_VAL
assert rec_te_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_TEST

assert sum_tr_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_TRAIN
assert sum_va_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_VAL
assert sum_te_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_TEST

# internal consistency for summarization anchor
train_qeq = float((sum_tr["qid"].astype(int) == sum_tr["src_nid"].astype(int)).mean())
val_qeq   = float((sum_va["qid"].astype(int) == sum_va["src_nid"].astype(int)).mean())
test_qeq  = float((sum_te["qid"].astype(int) == sum_te["src_nid"].astype(int)).mean())

print("INTERNAL summarization anchor consistency:")
print("train rate(qid == src_nid):", round(train_qeq, 6))
print("val   rate(qid == src_nid):", round(val_qeq, 6))
print("test  rate(qid == src_nid):", round(test_qeq, 6))

assert train_qeq == 1.0 and val_qeq == 1.0 and test_qeq == 1.0, "Internal summarization qid must equal src_nid."

print("Internal leakage and anchor checks passed.")

In [ ]:
# ============================================================
# CELL 6 — Corrected EXTERNAL rebuild from user-local summary slot
# IMPORTANT FIX:
#   ext_main_df['summaries'] is treated as a per-user summary slot/index,
#   then matched to the TRUE external summary row in ext_sum_df.
#   Final external qid = src_nid = matched NewsID.
# ============================================================
external_sum_lookup = {}
ext_eval_df = None

if ext_main_df is None or ext_sum_df is None or ext_sum_pkl is None:
    print("External files not loaded; skipping corrected external rebuild.")
else:
    def pick_ci(df, names):
        cmap = {c.lower(): c for c in df.columns}
        for n in names:
            if n.lower() in cmap:
                return cmap[n.lower()]
        raise KeyError(f"missing any of {names}; columns={list(df.columns)}")

    USER_COL_EXT = pick_ci(ext_main_df, ["userid", "user_id", "UserID"])
    DOCS_COL_EXT = pick_ci(ext_main_df, ["docs", "history", "documents"])
    ACTS_COL_EXT = pick_ci(ext_main_df, ["actions", "acts"])
    SUMS_COL_EXT = pick_ci(ext_main_df, ["summaries", "summary_id", "sid", "SummID", "summid"])

    SUMID_COL_EXTSUM  = pick_ci(ext_sum_df, ["summid", "SummID", "sid"])
    NEWSID_COL_EXTSUM = pick_ci(ext_sum_df, ["NewsID", "newsid"])
    USERID_COL_EXTSUM = pick_ci(ext_sum_df, ["userid", "UserID", "user_id"])
    TEXT_COL_EXTSUM   = pick_ci(ext_sum_df, ["extended_summary", "summary", "text"])

    def parse_list_like_cell(x):
        if isinstance(x, list):
            return x
        s = str(x)
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                return obj
        except Exception:
            pass
        return None

    def parse_docs_cell(x):
        obj = parse_list_like_cell(x)
        if obj is not None:
            return [normalize_news_id(z) for z in obj if normalize_news_id(z) is not None]
        return [m.upper() for m in NEWSID_RE.findall(str(x))]

    def parse_actions_cell(x):
        obj = parse_list_like_cell(x)
        if obj is not None:
            return [normalize_action(z) for z in obj]
        toks = re.findall(r"[A-Za-z_]+", str(x))
        return [normalize_action(z) for z in toks]

    def build_history_pairs(docs_cell, actions_cell):
        docs = parse_docs_cell(docs_cell)
        acts = parse_actions_cell(actions_cell)
        if len(acts) < len(docs):
            acts = acts + ["neu"] * (len(docs) - len(acts))
        elif len(acts) > len(docs):
            acts = acts[:len(docs)]
        return list(zip(docs, acts))

    def parse_summary_slot(x):
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        m = re.search(r"(\d+)", s)
        return int(m.group(1)) if m else np.nan

    # -------------------------
    # Normalize external summary table (raw order preserved first for embedding mapping)
    # -------------------------
    ext_sum_df = ext_sum_df.copy().reset_index(drop=True)
    ext_sum_df["sid_true"]      = ext_sum_df[SUMID_COL_EXTSUM].apply(normalize_sid)
    ext_sum_df["sid_num"]       = ext_sum_df["sid_true"].apply(sid_to_num)
    ext_sum_df["newsid_norm"]   = ext_sum_df[NEWSID_COL_EXTSUM].apply(normalize_news_id)
    ext_sum_df["userid_norm"]   = ext_sum_df[USERID_COL_EXTSUM].astype(str)
    ext_sum_df["gold_text"]     = ext_sum_df[TEXT_COL_EXTSUM].astype(str)
    ext_sum_df["gold_text_norm"] = (
        ext_sum_df["gold_text"].astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    # -------------------------
    # Build external embedding lookup keyed by TRUE sid
    # -------------------------
    ext_emb_lookup = {}

    def extract_emb(v):
        if v is None:
            return None
        if isinstance(v, dict):
            for kk in ["emb", "embedding", "sum_emb", "summary_emb", "t5_emb"]:
                if kk in v:
                    arr = np.asarray(v[kk], dtype=np.float32)
                    if arr.ndim == 1:
                        return arr
            return None
        arr = np.asarray(v, dtype=np.float32)
        return arr if arr.ndim == 1 else None

    if isinstance(ext_sum_pkl, dict):
        for k, v in ext_sum_pkl.items():
            sid = normalize_sid(k)
            arr = extract_emb(v)
            if sid is not None and arr is not None:
                ext_emb_lookup[sid] = arr
    elif isinstance(ext_sum_pkl, (list, tuple)):
        if len(ext_sum_pkl) != len(ext_sum_df):
            raise ValueError(f"ext_sum_pkl length {len(ext_sum_pkl)} != ext_sum_df length {len(ext_sum_df)}")
        for i, v in enumerate(ext_sum_pkl):
            sid = ext_sum_df.iloc[i]["sid_true"]
            arr = extract_emb(v)
            if sid is not None and arr is not None:
                ext_emb_lookup[sid] = arr
    else:
        raise TypeError(type(ext_sum_pkl))

    # -------------------------
    # Build user-local summary slot in external summary table
    # -------------------------
    ext_sum_df = ext_sum_df.sort_values(["userid_norm", "sid_num", "sid_true"]).reset_index(drop=True)
    ext_sum_df["summary_slot_user"] = ext_sum_df.groupby("userid_norm").cumcount() + 1

    # -------------------------
    # Build corrected external_sum_lookup keyed by TRUE sid
    # -------------------------
    for _, r in ext_sum_df.iterrows():
        sid = r["sid_true"]
        emb = ext_emb_lookup.get(sid, None)
        if emb is None:
            emb = np.zeros((sum_emb_dim,), dtype=np.float32)

        external_sum_lookup[sid] = {
            "newsid": r["newsid_norm"],
            "userid": r["userid_norm"],
            "gold_text": r["gold_text"],
            "emb": emb,
        }

    # -------------------------
    # Normalize external main table and join to TRUE summary row
    # -------------------------
    ext_main_df = ext_main_df.copy().reset_index(drop=True)
    ext_main_df["userid_norm"] = ext_main_df[USER_COL_EXT].astype(str)
    ext_main_df["summary_slot_user"] = ext_main_df[SUMS_COL_EXT].apply(parse_summary_slot)

    tqdm.pandas(desc="parsing external history")
    ext_main_df["history_parsed"] = ext_main_df.progress_apply(
        lambda r: build_history_pairs(r[DOCS_COL_EXT], r[ACTS_COL_EXT]),
        axis=1
    )

    ext_sum_join = ext_sum_df[[
        "userid_norm", "summary_slot_user", "sid_true", "newsid_norm", "gold_text"
    ]].copy()

    ext_main_df = ext_main_df.merge(
        ext_sum_join,
        on=["userid_norm", "summary_slot_user"],
        how="left",
        validate="many_to_one"
    )

    match_rate = float(ext_main_df["sid_true"].notna().mean())
    print("external join match rate:", round(match_rate, 6))
    print("matched rows:", int(ext_main_df["sid_true"].notna().sum()), "/", len(ext_main_df))

    # -------------------------
    # Build corrected external eval dataframe
    # -------------------------
    keep_rows = []
    for _, r in tqdm(ext_main_df.iterrows(), total=len(ext_main_df), desc="building corrected ext_eval_df"):
        sid = r["sid_true"]
        if pd.isna(sid):
            continue
        sid = str(sid)

        if sid not in external_sum_lookup:
            continue

        target_newsid = external_sum_lookup[sid]["newsid"]
        target_userid = external_sum_lookup[sid]["userid"]

        # strict consistency: joined user must match target user
        if str(r["userid_norm"]) != str(target_userid):
            continue

        hist_clean = strip_terminal_target_from_history(r["history_parsed"], target_newsid)
        h_ids, h_act, h_len = encode_history(hist_clean, MAX_HIS_LEN)

        src_nid = news2id.get(target_newsid, news2id["<UNK>"])
        qid = src_nid

        keep_rows.append({
            "userid": str(r["userid_norm"]),
            "sid": sid,
            "h_ids": h_ids,
            "h_act": h_act,
            "h_len": h_len,
            "src_nid": int(src_nid),
            "qid": int(qid),
        })

    ext_eval_df = pd.DataFrame(keep_rows)
    print("corrected ext_eval_df:", ext_eval_df.shape)

    # -------------------------
    # Tokenize ALL EXTERNAL gold summaries using corrected truth table
    # -------------------------
    EXT_TARGET_CACHE = {}
    for sid, info in tqdm(external_sum_lookup.items(), total=len(external_sum_lookup), desc="tokenizing external targets"):
        ids, mask = tok_text(info["gold_text"], MAX_TGT_TOK)
        EXT_TARGET_CACHE[sid] = {
            "ids": ids,
            "mask": mask,
            "text": info["gold_text"],
            "emb": info["emb"],
            "src_nid": news2id.get(info["newsid"], news2id["<UNK>"]),
            "newsid": info["newsid"],
            "userid": info["userid"],
        }

    # -------------------------
    # Strong sanity checks: corrected external structure
    # -------------------------
    chk = ext_eval_df.copy().reset_index(drop=True)
    chk["gold_text"] = chk["sid"].map(lambda s: external_sum_lookup[str(s)]["gold_text"])
    chk["gold_text_norm"] = (
        chk["gold_text"].astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    print("\n===== corrected external structure =====")
    print("rows:", len(chk))
    print("unique src_nid:", chk["src_nid"].nunique())
    print("unique qid:", chk["qid"].nunique())
    print("unique sid:", chk["sid"].nunique())
    print("unique users:", chk["userid"].nunique())
    print("unique normalized texts:", chk["gold_text_norm"].nunique())
    print("rate(qid == src_nid):", float((chk["qid"] == chk["src_nid"]).mean()))

    doc_user_summary = (
        chk.groupby("src_nid")
           .agg(
               n_rows=("src_nid", "size"),
               n_users=("userid", "nunique"),
               n_sids=("sid", "nunique"),
               n_texts=("gold_text_norm", "nunique"),
           )
           .reset_index()
           .sort_values(["n_users", "n_sids", "n_texts"], ascending=False)
    )

    print("docs with >=2 users:", int((doc_user_summary["n_users"] >= 2).sum()))
    print("docs with >=2 users and >=2 sids:", int(((doc_user_summary["n_users"] >= 2) & (doc_user_summary["n_sids"] >= 2)).sum()))
    print("docs with >=2 users and >=2 distinct texts:", int(((doc_user_summary["n_users"] >= 2) & (doc_user_summary["n_texts"] >= 2)).sum()))
    print("docs with >=2 users but only 1 distinct text:", int(((doc_user_summary["n_users"] >= 2) & (doc_user_summary["n_texts"] == 1)).sum()))

    # external leakage sanity
    ext_stats = leakage_stats(ext_eval_df, "ext_eval_df")
    assert ext_stats["bad_hlen_count"] == 0, f"external has invalid h_len rows: {ext_stats['bad_hlen_count']}"
    assert ext_stats["last_item_equals_qid_rate"] <= MAX_ACCEPTABLE_LAST_EQ_RATE_TEST, \
        f"external leakage suspiciously high: {ext_stats['last_item_equals_qid_rate']:.6f}"

    print("Corrected external rebuild passed.")

In [ ]:
# ============================================================
# CELL 7 — Dataset classes + loaders + helper diagnostics
# Includes subjectivity checks showing PerSEval feasibility
# ============================================================
def _norm_text_for_subjectivity(x):
    s = "" if x is None else str(x)
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def subjectivity_report(df, sid_to_text_fn, split_name, anchor_col="qid", user_col="userid"):
    tmp = df.copy().reset_index(drop=True)
    tmp["gold_text"] = tmp["sid"].astype(str).map(sid_to_text_fn)
    tmp["gold_text_norm"] = tmp["gold_text"].map(_norm_text_for_subjectivity)

    tmp[anchor_col] = pd.to_numeric(tmp[anchor_col], errors="coerce").fillna(-1).astype(int)
    tmp[user_col] = tmp[user_col].astype(str)

    grp = (
        tmp.groupby(anchor_col)
           .agg(
               n_rows=("sid", "size"),
               n_users=(user_col, "nunique"),
               n_sids=("sid", "nunique"),
               n_texts=("gold_text_norm", "nunique"),
           )
           .reset_index()
           .rename(columns={anchor_col: "anchor_id"})
    )

    grp["valid_subjective_anchor"] = ((grp["n_users"] >= 2) & (grp["n_texts"] >= 2)).astype(int)

    print(f"\n===== {split_name} | anchor={anchor_col} =====")
    print("anchors total:", int(len(grp)))
    print("anchors with >=2 users:", int((grp["n_users"] >= 2).sum()))
    print("anchors with >=2 distinct texts:", int((grp["n_texts"] >= 2).sum()))
    print("anchors with BOTH >=2 users and >=2 distinct texts:", int(grp["valid_subjective_anchor"].sum()))

    if len(grp) > 0:
        print("share valid subjective anchors:", round(float(grp["valid_subjective_anchor"].mean()), 6))

    return grp

# -------------------------
# Internal subjectivity checks
# qid == src_nid by construction for summarization
# -------------------------
internal_sid_to_text = lambda s: SUM_TARGET_CACHE[str(s)]["text"] if str(s) in SUM_TARGET_CACHE else None

train_subj = subjectivity_report(sum_tr, internal_sid_to_text, "INTERNAL TRAIN", anchor_col="qid", user_col="userid")
val_subj   = subjectivity_report(sum_va, internal_sid_to_text, "INTERNAL VAL",   anchor_col="qid", user_col="userid")
test_subj  = subjectivity_report(sum_te, internal_sid_to_text, "INTERNAL TEST",  anchor_col="qid", user_col="userid")

# -------------------------
# External subjectivity checks
# -------------------------
if ext_eval_df is not None and len(ext_eval_df) > 0:
    external_sid_to_text = lambda s: external_sum_lookup[str(s)]["gold_text"] if str(s) in external_sum_lookup else None
    ext_subj = subjectivity_report(ext_eval_df, external_sid_to_text, "EXTERNAL TEST", anchor_col="qid", user_col="userid")
else:
    ext_subj = None
    print("\nExternal subjectivity report skipped because ext_eval_df is unavailable.")

# -------------------------
# Dataset classes
# -------------------------
class RecDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {
            "task_id": torch.tensor(TASK2ID["rec"], dtype=torch.long),
            "h_ids": torch.tensor(r["h_ids"], dtype=torch.long),
            "h_act": torch.tensor(r["h_act"], dtype=torch.long),
            "h_len": torch.tensor(int(r["h_len"]), dtype=torch.long),
            "qid": torch.tensor(int(r["qid"]), dtype=torch.long),
        }

class SumDataset(Dataset):
    """
    Internal summarization dataset.
    qid == src_nid by construction.
    sid is the gold summary target id.
    """
    def __init__(self, df, target_cache):
        self.df = df.reset_index(drop=True)
        self.target_cache = target_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        sid = str(r["sid"])
        s = self.target_cache[sid]
        return {
            "task_id": torch.tensor(TASK2ID["sum"], dtype=torch.long),
            "h_ids": torch.tensor(r["h_ids"], dtype=torch.long),
            "h_act": torch.tensor(r["h_act"], dtype=torch.long),
            "h_len": torch.tensor(int(r["h_len"]), dtype=torch.long),
            "qid": torch.tensor(int(r["qid"]), dtype=torch.long),
            "src_nid": torch.tensor(int(r["src_nid"]), dtype=torch.long),
            "sid": sid,
            "tgt_ids": torch.tensor(s["ids"], dtype=torch.long),
            "tgt_mask": torch.tensor(s["mask"], dtype=torch.long),
            "tgt_emb": torch.tensor(s["emb"], dtype=torch.float32),
            "gold_text": s["text"],
            "userid": str(r["userid"]),
        }

class ExtDataset(Dataset):
    """
    Corrected external summarization dataset.
    sid is TRUE external summary id from ext_sum_df after corrected join.
    qid == src_nid by construction.
    """
    def __init__(self, df, target_cache):
        self.df = df.reset_index(drop=True)
        self.target_cache = target_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        sid = str(r["sid"])
        s = self.target_cache[sid]

        return {
            "task_id": torch.tensor(TASK2ID["sum"], dtype=torch.long),
            "h_ids": torch.tensor(r["h_ids"], dtype=torch.long),
            "h_act": torch.tensor(r["h_act"], dtype=torch.long),
            "h_len": torch.tensor(int(r["h_len"]), dtype=torch.long),
            "qid": torch.tensor(int(r["qid"]), dtype=torch.long),
            "src_nid": torch.tensor(int(r["src_nid"]), dtype=torch.long),
            "sid": sid,
            "tgt_ids": torch.tensor(s["ids"], dtype=torch.long),
            "tgt_mask": torch.tensor(s["mask"], dtype=torch.long),
            "tgt_emb": torch.tensor(s["emb"], dtype=torch.float32),
            "gold_text": s["text"],
            "userid": s["userid"],
        }

def dict_collate(batch):
    out = {}
    keys = batch[0].keys()
    for k in keys:
        vals = [x[k] for x in batch]
        v0 = vals[0]
        if torch.is_tensor(v0):
            out[k] = torch.stack(vals, dim=0)
        elif isinstance(v0, np.ndarray):
            out[k] = torch.stack([torch.as_tensor(v) for v in vals], dim=0)
        elif isinstance(v0, (int, np.integer)):
            out[k] = torch.tensor(vals, dtype=torch.long)
        elif isinstance(v0, (float, np.floating)):
            out[k] = torch.tensor(vals, dtype=torch.float32)
        else:
            out[k] = vals
    return out

pin = (device.type == "cuda")

# -------------------------
# Loaders
# -------------------------
rec_train_loader = DataLoader(
    RecDataset(rec_tr),
    batch_size=REC_BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    collate_fn=dict_collate,
    persistent_workers=(NUM_WORKERS > 0),
)

rec_val_loader = DataLoader(
    RecDataset(rec_va),
    batch_size=REC_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    collate_fn=dict_collate,
    persistent_workers=(NUM_WORKERS > 0),
)

rec_test_loader = DataLoader(
    RecDataset(rec_te),
    batch_size=REC_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    collate_fn=dict_collate,
    persistent_workers=(NUM_WORKERS > 0),
)

sum_train_loader = DataLoader(
    SumDataset(sum_tr, SUM_TARGET_CACHE),
    batch_size=SUM_BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    collate_fn=dict_collate,
    persistent_workers=(NUM_WORKERS > 0),
)

sum_val_loader = DataLoader(
    SumDataset(sum_va, SUM_TARGET_CACHE),
    batch_size=VAL_SUM_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    collate_fn=dict_collate,
    persistent_workers=(NUM_WORKERS > 0),
)

sum_test_loader = DataLoader(
    SumDataset(sum_te, SUM_TARGET_CACHE),
    batch_size=VAL_SUM_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    collate_fn=dict_collate,
    persistent_workers=(NUM_WORKERS > 0),
)

ext_loader = None
if ext_eval_df is not None and len(ext_eval_df) > 0 and len(EXT_TARGET_CACHE) > 0:
    ext_loader = DataLoader(
        ExtDataset(ext_eval_df, EXT_TARGET_CACHE),
        batch_size=VAL_SUM_BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        collate_fn=dict_collate,
        persistent_workers=(NUM_WORKERS > 0),
    )

# -------------------------
# Basic helper functions used by later model/training cells
# -------------------------
def to_device(batch):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        elif isinstance(v, np.ndarray):
            out[k] = torch.as_tensor(v).to(device, non_blocking=True)
        elif isinstance(v, (list, tuple)) and len(v) > 0 and isinstance(v[0], (int, np.integer)):
            out[k] = torch.tensor(v, dtype=torch.long, device=device)
        elif isinstance(v, (list, tuple)) and len(v) > 0 and isinstance(v[0], (float, np.floating)):
            out[k] = torch.tensor(v, dtype=torch.float32, device=device)
        else:
            out[k] = v
    return out

def make_mask(lengths, T):
    t = torch.arange(T, device=lengths.device).unsqueeze(0)
    return t < lengths.unsqueeze(1)

def last_valid_state(seq, lengths):
    idx = (lengths - 1).clamp_min(0)
    return seq[torch.arange(seq.size(0), device=seq.device), idx]

def masked_mean(x, mask, dim=1, eps=1e-8):
    w = mask.float().unsqueeze(-1)
    return (x * w).sum(dim=dim) / w.sum(dim=dim).clamp_min(eps)

def sample_negs(pos_ids, num_items, k, generator, min_id=2):
    B = pos_ids.shape[0]
    neg = torch.randint(min_id, num_items, (B, k), generator=generator, device=pos_ids.device)
    pos_exp = pos_ids[:, None].expand_as(neg)
    coll = neg.eq(pos_exp)
    if coll.any():
        neg[coll] = ((neg[coll] + 1 - min_id) % max(1, (num_items - min_id))) + min_id
        coll = neg.eq(pos_exp)
        if coll.any():
            neg[coll] = min_id
    return neg

def rec_metrics_from_ranks(ranks, Ks=(5, 10)):
    ranks = ranks.float()
    out = {"MRR": (1.0 / ranks).mean().item()}
    for K in Ks:
        hit = (ranks <= K).float().mean().item()
        ndcg = torch.zeros_like(ranks)
        ok = ranks <= K
        ndcg[ok] = 1.0 / torch.log2(ranks[ok] + 1.0)
        out[f"Hit@{K}"] = hit
        out[f"NDCG@{K}"] = ndcg.mean().item()
    return out

print("loaders ready")
print("internal train subjective anchors:", int(train_subj["valid_subjective_anchor"].sum()))
print("internal val subjective anchors  :", int(val_subj["valid_subjective_anchor"].sum()))
print("internal test subjective anchors :", int(test_subj["valid_subjective_anchor"].sum()))
if ext_subj is not None:
    print("external test subjective anchors :", int(ext_subj["valid_subjective_anchor"].sum()))

In [ ]:
# ============================================================
# CELL 8 — PerTIDE encoder: tri-state SSM over history only
# ============================================================
class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)


class ActionGate(nn.Module):
    """
    Action-conditioned gate mu_t in the encoder.
    History only. No target leakage.
    """
    def __init__(self, num_actions, d_model):
        super().__init__()
        self.num_actions = num_actions
        self.gate = nn.Sequential(
            nn.Linear(num_actions, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, act_ids):
        x = F.one_hot(act_ids, num_classes=self.num_actions).float()
        return torch.sigmoid(self.gate(x))


class OrthogonalBasis(nn.Module):
    """
    Learnable orthogonal basis Q used by the tri-state diagonal dynamics.
    """
    def __init__(self, d_model):
        super().__init__()
        self.raw = nn.Parameter(torch.randn(d_model, d_model) / math.sqrt(d_model))

    def forward(self):
        q, _ = torch.linalg.qr(self.raw.float())
        return q.to(self.raw.dtype)


class TriStateSSM(nn.Module):
    """
    Tri-state SSM:
      long-term memory   m^L
      short-term memory  m^S
      event memory       m^E
    Processes only the history sequence b_t.
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.basis = OrthogonalBasis(d_model)

        self.lambda_long_raw  = nn.Parameter(torch.randn(d_model) * 0.02)
        self.lambda_short_raw = nn.Parameter(torch.randn(d_model) * 0.02)
        self.lambda_event_raw = nn.Parameter(torch.randn(d_model) * 0.02)

        self.delta_raw = nn.Parameter(torch.tensor(1.0))
        self.kappa_raw = nn.Parameter(torch.tensor(1.0))
        self.eta_raw   = nn.Parameter(torch.tensor(0.1))

        self.B = nn.Linear(d_model, d_model, bias=False)

    def _apply_A(self, x, a_diag, Q):
        x_hat = x @ Q
        x_hat = x_hat * a_diag.unsqueeze(0)
        return x_hat @ Q.T

    def forward(self, b_seq, mask):
        B, T, D = b_seq.shape
        Q = self.basis()

        lam_long  = -F.softplus(self.lambda_long_raw)
        lam_short =  F.softplus(self.lambda_short_raw)
        lam_event = -F.softplus(self.lambda_event_raw)

        delta = F.softplus(self.delta_raw)
        kappa = F.softplus(self.kappa_raw)
        eta   = F.softplus(self.eta_raw)

        a_long  = torch.exp(delta * lam_long).clamp(1e-5, 0.99995)
        a_short = torch.sigmoid(-kappa * lam_short).clamp(1e-5, 0.99995)
        a_event = (1.0 + eta * lam_event).clamp(1e-5, 0.99995)

        mL = torch.zeros((B, D), device=b_seq.device, dtype=b_seq.dtype)
        mS = torch.zeros((B, D), device=b_seq.device, dtype=b_seq.dtype)
        mE = torch.zeros((B, D), device=b_seq.device, dtype=b_seq.dtype)

        outsL, outsS, outsE = [], [], []
        for t in range(T):
            valid = mask[:, t].unsqueeze(-1).float()
            u_t = self.B(b_seq[:, t])

            mL_new = self._apply_A(mL, a_long, Q) + u_t
            mS_new = self._apply_A(mS, a_short, Q) + u_t
            mE_new = self._apply_A(mE, a_event, Q) + u_t

            mL = valid * mL_new + (1.0 - valid) * mL
            mS = valid * mS_new + (1.0 - valid) * mS
            mE = valid * mE_new + (1.0 - valid) * mE

            outsL.append(mL)
            outsS.append(mS)
            outsE.append(mE)

        return (
            torch.stack(outsL, dim=1),
            torch.stack(outsS, dim=1),
            torch.stack(outsE, dim=1),
        )


class PerTIDEEncoder(nn.Module):
    """
    Encoder consumes only:
      history item embeddings
      history action sequence
      history lengths
      task ids

    It does NOT consume target summaries or future targets.
    """
    def __init__(self, item_emb_dim, d_model, num_actions, num_tasks, max_len):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.num_tasks = num_tasks

        self.item_proj = nn.Sequential(
            nn.LayerNorm(item_emb_dim),
            nn.Linear(item_emb_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.action_gate = ActionGate(num_actions, d_model)
        self.ssm = TriStateSSM(d_model)

        self.fuse = MLP(3 * d_model, d_model, d_model, dropout=DROPOUT)
        self.psi_update = nn.Linear(d_model, d_model)

        # task-conditioned routing, faithful to the earlier formulation
        self.task_emb = nn.Embedding(num_tasks, d_model)
        self.task_dist = nn.Linear(2 * d_model, num_tasks)
        self.task_to_state = nn.Linear(num_tasks, d_model, bias=False)
        self.bG = nn.Parameter(torch.zeros(d_model))

        # exact auxiliary heads / losses
        self.pos_head = nn.Linear(d_model, max_len)
        self.next_proj = nn.Linear(d_model, d_model)
        self.task_cls = nn.Linear(d_model, num_tasks)

    def project_items(self, item_emb):
        return self.item_proj(item_emb)

    def encode_history(self, hist_item_emb, hist_act, hist_len, task_ids):
        mask = make_mask(hist_len, hist_item_emb.size(1))

        # e_t
        e = self.project_items(hist_item_emb)

        # b_t = mu_t ⊙ e_t
        mu = self.action_gate(hist_act)
        b = mu * e

        # tri-state history processing
        mL, mS, mE = self.ssm(b, mask)

        # fused hidden pre-state
        h_pre = self.fuse(torch.cat([mL, mS, mE], dim=-1)) + self.psi_update(b)

        # task-conditioned dynamic routing
        gq = self.task_emb(task_ids)
        z_prev = torch.zeros(
            (hist_item_emb.size(0), self.d_model),
            device=hist_item_emb.device,
            dtype=hist_item_emb.dtype
        )

        z_seq = []
        pi_seq = []
        for t in range(hist_item_emb.size(1)):
            pi_logits_t = self.task_dist(torch.cat([z_prev, gq], dim=-1))
            pi_t = torch.softmax(pi_logits_t, dim=-1)

            z_t = self.task_to_state(pi_t) * h_pre[:, t] + self.bG
            valid = mask[:, t].unsqueeze(-1).float()
            z_t = valid * z_t + (1.0 - valid) * z_prev

            z_seq.append(z_t)
            pi_seq.append(pi_t)
            z_prev = z_t

        z_seq = torch.stack(z_seq, dim=1)
        pi_seq = torch.stack(pi_seq, dim=1)

        z_last = last_valid_state(z_seq, hist_len)
        z_hat_q = self.next_proj(z_last)

        return {
            "mask": mask,
            "e": e,
            "b": b,
            "mL": mL,
            "mS": mS,
            "mE": mE,
            "h_pre": h_pre,
            "z_seq": z_seq,
            "z_last": z_last,
            "z_hat_q": z_hat_q,
            "pi_seq": pi_seq,
            "task_logits": self.task_cls(z_last),
            "pos_logits": self.pos_head(z_seq),
        }

    def encoder_losses(self, enc_out, task_ids, q_item_ids, item_table_proj, hist_len):
        """
        Exact encoder losses:
          Lenc = 0.2 Lpos + 0.4 Lnext + 0.4 Ltask
        """
        mask = enc_out["mask"]
        z_seq = enc_out["z_seq"]
        z_hat_q = enc_out["z_hat_q"]

        # Lpos
        B, T, _ = z_seq.shape
        pos_targets = torch.arange(T, device=z_seq.device).unsqueeze(0).expand(B, T)
        pos_logits = enc_out["pos_logits"]
        l_pos = (
            F.cross_entropy(pos_logits[mask], pos_targets[mask])
            if mask.any()
            else torch.tensor(0.0, device=z_seq.device)
        )

        # Ltask
        l_task = F.cross_entropy(enc_out["task_logits"], task_ids)

        # Lnext with sampled softmax over item vocabulary
        gen = torch.Generator(device=z_seq.device).manual_seed(SEED + int(hist_len.sum().item()))
        neg = sample_negs(q_item_ids, item_table_proj.size(0), ENC_NEXT_NEGS, gen)
        cand = torch.cat([q_item_ids[:, None], neg], dim=1)
        cand_vec = item_table_proj.index_select(0, cand.reshape(-1)).view(cand.size(0), cand.size(1), -1)
        logits = torch.einsum("bd,bkd->bk", z_hat_q, cand_vec)
        l_next = F.cross_entropy(
            logits,
            torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
        )

        l_enc = (LAMBDA_POS * l_pos) + (LAMBDA_NEXT * l_next) + (LAMBDA_TASK * l_task)

        return {
            "l_enc": l_enc,
            "l_pos": l_pos,
            "l_next": l_next,
            "l_task": l_task,
        }

print("PerTIDE Task Agnostic Encoder ready")

In [ ]:
# ============================================================
# CELL 9 — UserScore-style recommendation head + UGI-conditioned DistilGPT2 decoder
# ============================================================
class UserScoreStyleRecHead(nn.Module):
    """
    UserScore-style scoring:
      user/history representation · candidate news representation
    """
    def __init__(self):
        super().__init__()

    def forward(self, user_vec, cand_item_proj):
        return torch.einsum("bd,bkd->bk", user_vec, cand_item_proj)


class UGIDistilGPT2Decoder(nn.Module):
    """
    DistilGPT2 decoder conditioned by:
      z_hat_q : encoder query-target representation
      z_user  : encoder user/history representation
      e_dq    : query-document embedding
    with UGI gating.
    """
    def __init__(self, decoder_name, d_model, sum_emb_dim, prefix_len=8, use_ugi=True, dropout=0.1):
        super().__init__()
        self.use_ugi = use_ugi
        self.prefix_len = prefix_len

        self.lm = AutoModelForCausalLM.from_pretrained(decoder_name)
        self.lm_dim = self.lm.config.n_embd
        self.lm.config.use_cache = False

        if self.lm.config.pad_token_id is None:
            self.lm.config.pad_token_id = tokenizer.pad_token_id

        if hasattr(self.lm, "gradient_checkpointing_enable"):
            self.lm.gradient_checkpointing_enable()

        # personalization / contextual gating
        self.ufi = MLP(2 * d_model, d_model, d_model, dropout=dropout)
        self.gamma = MLP(d_model, d_model, d_model, dropout=dropout)
        self.beta  = MLP(d_model, d_model, d_model, dropout=dropout)
        self.ugi   = MLP(2 * d_model, d_model, d_model, dropout=dropout)

        # stronger query-document grounding
        self.doc_gate = MLP(2 * d_model, d_model, d_model, dropout=dropout)
        self.prefix_ln = nn.LayerNorm(d_model)
        self.prefix_drop = nn.Dropout(dropout)
        self.to_prefix = nn.Linear(d_model, prefix_len * self.lm_dim)

        # semantic supervision
        self.sem_head = nn.Sequential(
            nn.LayerNorm(self.lm_dim),
            nn.Linear(self.lm_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, sum_emb_dim),
        )

        self.freeze_all_lm()

    def freeze_all_lm(self):
        for p in self.lm.parameters():
            p.requires_grad = False

    def unfreeze_top_lm(self, n_blocks=2, train_ln_f=True):
        self.freeze_all_lm()
        if n_blocks > 0:
            blocks = self.lm.transformer.h
            for blk in blocks[-n_blocks:]:
                for p in blk.parameters():
                    p.requires_grad = True
        if train_ln_f and hasattr(self.lm.transformer, "ln_f"):
            for p in self.lm.transformer.ln_f.parameters():
                p.requires_grad = True

    def build_control(self, z_hat_q, z_user, e_dq):
        if self.use_ugi:
            gamma = self.gamma(z_user)
            beta = self.beta(z_user)
            z_tilde = gamma * z_hat_q + beta
            c_user = self.ugi(torch.cat([z_tilde, e_dq], dim=-1))
        else:
            c_user = self.ufi(torch.cat([z_hat_q, e_dq], dim=-1))

        doc_ctx = self.doc_gate(torch.cat([c_user, e_dq], dim=-1))
        c = self.prefix_ln(c_user + doc_ctx)
        prefix = self.to_prefix(self.prefix_drop(c)).view(c.size(0), self.prefix_len, self.lm_dim)
        return prefix

    def forward_lm(self, z_hat_q, z_user, e_dq, src_ids, src_mask, tgt_ids, tgt_mask):
        prefix = self.build_control(z_hat_q, z_user, e_dq)

        tgt_in = tgt_ids[:, :-1]
        tgt_out = tgt_ids[:, 1:]
        tgt_mask_in = tgt_mask[:, :-1]
        tgt_mask_out = tgt_mask[:, 1:]

        prompt_emb = self.lm.transformer.wte(src_ids)
        tgt_emb = self.lm.transformer.wte(tgt_in)

        inputs_embeds = torch.cat([prefix, prompt_emb, tgt_emb], dim=1)

        prefix_mask = torch.ones(
            (src_ids.size(0), self.prefix_len),
            device=src_ids.device,
            dtype=src_mask.dtype,
        )
        attn_mask = torch.cat([prefix_mask, src_mask, tgt_mask_in], dim=1)

        out = self.lm(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask,
            output_hidden_states=True,
            use_cache=False,
        )

        logits = out.logits
        target_logits = logits[:, -tgt_in.size(1):, :].contiguous().view(-1, logits.size(-1))
        target_labels = tgt_out.contiguous().view(-1)

        ce = F.cross_entropy(
            target_logits,
            target_labels,
            ignore_index=tokenizer.pad_token_id,
        )

        hidden = out.hidden_states[-1][:, -tgt_in.size(1):, :]
        return {
            "ce": ce,
            "hidden": hidden,
            "tgt_mask_out": tgt_mask_out,
        }

    def loss(self, z_hat_q, z_user, e_dq, src_ids, src_mask, tgt_ids, tgt_mask, tgt_emb=None, sem_weight=0.0):
        lm_out = self.forward_lm(
            z_hat_q=z_hat_q,
            z_user=z_user,
            e_dq=e_dq,
            src_ids=src_ids,
            src_mask=src_mask,
            tgt_ids=tgt_ids,
            tgt_mask=tgt_mask,
        )

        ce = lm_out["ce"]
        total = ce
        sem = torch.tensor(0.0, device=ce.device)

        if tgt_emb is not None and sem_weight > 0:
            pooled = masked_mean(lm_out["hidden"], lm_out["tgt_mask_out"], dim=1)
            pred_sem = self.sem_head(pooled)
            valid = tgt_emb.norm(dim=-1) > 1e-8
            if valid.any():
                pred = F.normalize(pred_sem[valid], dim=-1)
                gold = F.normalize(tgt_emb[valid], dim=-1)
                sem = 1.0 - (pred * gold).sum(dim=-1).mean()
                total = total + sem_weight * sem

        return total, {
            "l_ce": ce.detach(),
            "l_sem": sem.detach(),
        }

    @torch.no_grad()
    def generate(self, z_hat_q, z_user, e_dq, src_ids, src_mask, max_new_tokens=48):
        prefix = self.build_control(z_hat_q, z_user, e_dq)
        prompt_emb = self.lm.transformer.wte(src_ids)
        inputs_embeds = torch.cat([prefix, prompt_emb], dim=1)

        prefix_mask = torch.ones(
            (src_ids.size(0), self.prefix_len),
            device=src_ids.device,
            dtype=src_mask.dtype,
        )
        attn_mask = torch.cat([prefix_mask, src_mask], dim=1)

        gen_ids = self.lm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
        return gen_ids


class PerTIDERecSum(nn.Module):
    def __init__(self, raw_item_dim, d_model, num_actions, num_tasks, max_len, decoder_name, sum_emb_dim, prefix_len=8, use_ugi=True):
        super().__init__()
        self.encoder = PerTIDEEncoder(
            item_emb_dim=raw_item_dim,
            d_model=d_model,
            num_actions=num_actions,
            num_tasks=num_tasks,
            max_len=max_len,
        )
        self.rec_head = UserScoreStyleRecHead()
        self.decoder = UGIDistilGPT2Decoder(
            decoder_name=decoder_name,
            d_model=d_model,
            sum_emb_dim=sum_emb_dim,
            prefix_len=prefix_len,
            use_ugi=use_ugi,
            dropout=DROPOUT,
        )

    def projected_item_table(self, item_emb_table):
        return self.encoder.project_items(item_emb_table)

    def rec_logits_from_candidates(self, z_hat_q, cand_item_proj):
        return self.rec_head(z_hat_q, cand_item_proj)


model = PerTIDERecSum(
    raw_item_dim=NEWS_EMB_T.size(1),
    d_model=D_MODEL,
    num_actions=NUM_ACTIONS,
    num_tasks=len(TASK_NAMES),
    max_len=MAX_HIS_LEN,
    decoder_name=DECODER_NAME,
    sum_emb_dim=sum_emb_dim,
    prefix_len=PREFIX_LEN,
    use_ugi=USE_UGI,
).to(device)

if USE_COMPILE and hasattr(torch, "compile"):
    model = torch.compile(model)

ITEM_EMB_CPU = NEWS_EMB_T.clone().float()

print("model ready")
print("encoder params:", sum(p.numel() for p in model.encoder.parameters()))
print("decoder params:", sum(p.numel() for p in model.decoder.parameters()))
print("distilgpt2 total params:", sum(p.numel() for p in model.decoder.lm.parameters()))
print("Task Heads ready")

In [ ]:
# ============================================================
# CELL 10 — Stage helpers, optimizers, losses, checkpoint helpers
# ============================================================
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def save_ckpt(path, payload):
    tmp = path + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, path)

def load_matching_state(module, state_dict):
    model_sd = module.state_dict()
    keep = {}
    for k, v in state_dict.items():
        if k in model_sd and tuple(model_sd[k].shape) == tuple(v.shape):
            keep[k] = v
    model_sd.update(keep)
    module.load_state_dict(model_sd)
    return sorted(keep.keys())

def build_projected_item_table():
    with torch.no_grad():
        item_proj = model.projected_item_table(ITEM_EMB_CPU.to(device))
        item_proj[0] = 0.0
    return item_proj

def gather_hist_item_emb(h_ids):
    flat = h_ids.reshape(-1).detach().cpu()
    out = ITEM_EMB_CPU.index_select(0, flat).view(h_ids.size(0), h_ids.size(1), -1)
    return out.to(device, non_blocking=True)

def gather_item_proj(item_proj_table, item_ids):
    flat = item_ids.reshape(-1)
    out = item_proj_table.index_select(0, flat).view(item_ids.size(0), item_ids.size(1), -1)
    return out

def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag

def configure_stage1_rec():
    """
    Stage 1:
      train encoder + rec head only
      decoder frozen
    """
    set_requires_grad(model, False)
    set_requires_grad(model.encoder, True)
    set_requires_grad(model.rec_head, True)
    model.decoder.freeze_all_lm()
    model.train()

def configure_stage2_joint():
    """
    Stage 2:
      train encoder + rec head + UGI/context + top DistilGPT2 blocks
    """
    set_requires_grad(model, False)

    set_requires_grad(model.encoder, True)
    set_requires_grad(model.rec_head, True)

    set_requires_grad(model.decoder.gamma, True)
    set_requires_grad(model.decoder.beta, True)
    set_requires_grad(model.decoder.ugi, True)
    set_requires_grad(model.decoder.ufi, True)
    set_requires_grad(model.decoder.doc_gate, True)
    set_requires_grad(model.decoder.sem_head, True)
    set_requires_grad(model.decoder.to_prefix, True)
    set_requires_grad(model.decoder.prefix_ln, True)

    model.decoder.unfreeze_top_lm(
        n_blocks=DECODER_TRAIN_LAST_N_BLOCKS,
        train_ln_f=TRAIN_DECODER_LN_F,
    )
    model.train()

def configure_stage3_sumft():
    """
    Stage 3:
      freeze encoder + rec head
      fine-tune only summary-side contextualizer + top DistilGPT2 blocks
    """
    set_requires_grad(model, False)

    set_requires_grad(model.decoder.gamma, True)
    set_requires_grad(model.decoder.beta, True)
    set_requires_grad(model.decoder.ugi, True)
    set_requires_grad(model.decoder.ufi, True)
    set_requires_grad(model.decoder.doc_gate, True)
    set_requires_grad(model.decoder.sem_head, True)
    set_requires_grad(model.decoder.to_prefix, True)
    set_requires_grad(model.decoder.prefix_ln, True)

    model.decoder.unfreeze_top_lm(
        n_blocks=DECODER_TRAIN_LAST_N_BLOCKS,
        train_ln_f=TRAIN_DECODER_LN_F,
    )
    model.train()

def named_params_by_prefix(module, prefix):
    return [(f"{prefix}.{n}", p) for n, p in module.named_parameters() if p.requires_grad]

def split_decay_groups(named_params, lr, weight_decay):
    decay, no_decay = [], []
    for n, p in named_params:
        if not p.requires_grad:
            continue
        name_l = n.lower()
        if p.ndim < 2 or n.endswith(".bias") or "norm" in name_l or "ln_" in name_l or "layernorm" in name_l:
            no_decay.append(p)
        else:
            decay.append(p)

    groups = []
    if decay:
        groups.append({"params": decay, "lr": lr, "weight_decay": weight_decay})
    if no_decay:
        groups.append({"params": no_decay, "lr": lr, "weight_decay": 0.0})
    return groups

def build_stage1_optimizer():
    configure_stage1_rec()
    named = []
    named += named_params_by_prefix(model.encoder, "encoder")
    named += named_params_by_prefix(model.rec_head, "rec_head")
    groups = split_decay_groups(named, STAGE1_REC_LR, WD)
    return torch.optim.AdamW(groups, betas=(0.9, 0.95))

def build_stage2_optimizer():
    configure_stage2_joint()

    enc_named = []
    enc_named += named_params_by_prefix(model.encoder, "encoder")
    enc_named += named_params_by_prefix(model.rec_head, "rec_head")

    ctx_named = []
    ctx_named += named_params_by_prefix(model.decoder.gamma, "decoder.gamma")
    ctx_named += named_params_by_prefix(model.decoder.beta, "decoder.beta")
    ctx_named += named_params_by_prefix(model.decoder.ugi, "decoder.ugi")
    ctx_named += named_params_by_prefix(model.decoder.ufi, "decoder.ufi")
    ctx_named += named_params_by_prefix(model.decoder.doc_gate, "decoder.doc_gate")
    ctx_named += named_params_by_prefix(model.decoder.sem_head, "decoder.sem_head")
    ctx_named += named_params_by_prefix(model.decoder.to_prefix, "decoder.to_prefix")
    ctx_named += named_params_by_prefix(model.decoder.prefix_ln, "decoder.prefix_ln")

    dec_named = []
    for i, blk in enumerate(model.decoder.lm.transformer.h):
        dec_named += named_params_by_prefix(blk, f"decoder.lm.transformer.h.{i}")
    if hasattr(model.decoder.lm.transformer, "ln_f"):
        dec_named += named_params_by_prefix(model.decoder.lm.transformer.ln_f, "decoder.lm.transformer.ln_f")

    groups = []
    groups += split_decay_groups(enc_named, STAGE2_ENCODER_LR, WD)
    groups += split_decay_groups(ctx_named, STAGE2_CONTEXT_LR, WD)
    groups += split_decay_groups(dec_named, STAGE2_DECODER_LR, WD)
    return torch.optim.AdamW(groups, betas=(0.9, 0.95))

def build_stage3_optimizer():
    configure_stage3_sumft()

    ctx_named = []
    ctx_named += named_params_by_prefix(model.decoder.gamma, "decoder.gamma")
    ctx_named += named_params_by_prefix(model.decoder.beta, "decoder.beta")
    ctx_named += named_params_by_prefix(model.decoder.ugi, "decoder.ugi")
    ctx_named += named_params_by_prefix(model.decoder.ufi, "decoder.ufi")
    ctx_named += named_params_by_prefix(model.decoder.doc_gate, "decoder.doc_gate")
    ctx_named += named_params_by_prefix(model.decoder.sem_head, "decoder.sem_head")
    ctx_named += named_params_by_prefix(model.decoder.to_prefix, "decoder.to_prefix")
    ctx_named += named_params_by_prefix(model.decoder.prefix_ln, "decoder.prefix_ln")

    dec_named = []
    for i, blk in enumerate(model.decoder.lm.transformer.h):
        dec_named += named_params_by_prefix(blk, f"decoder.lm.transformer.h.{i}")
    if hasattr(model.decoder.lm.transformer, "ln_f"):
        dec_named += named_params_by_prefix(model.decoder.lm.transformer.ln_f, "decoder.lm.transformer.ln_f")

    groups = []
    groups += split_decay_groups(ctx_named, STAGE3_CONTEXT_LR, WD)
    groups += split_decay_groups(dec_named, STAGE3_DECODER_LR, WD)
    return torch.optim.AdamW(groups, betas=(0.9, 0.95))

def build_warmup_cosine_scheduler(optimizer, total_steps, warmup_ratio):
    warmup_steps = max(1, int(total_steps * warmup_ratio))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def rec_step(batch, item_proj_table, generator):
    task_ids = batch["task_id"]
    h_ids = batch["h_ids"]
    h_act = batch["h_act"]
    h_len = batch["h_len"]
    qid = batch["qid"]

    hist_item_emb = gather_hist_item_emb(h_ids)

    with torch.cuda.amp.autocast(enabled=USE_AMP):
        enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)
        enc_losses = model.encoder.encoder_losses(enc_out, task_ids, qid, item_proj_table, h_len)

        neg = sample_negs(qid, item_proj_table.size(0), REC_TRAIN_NEGS, generator)
        cand = torch.cat([qid[:, None], neg], dim=1)
        cand_proj = gather_item_proj(item_proj_table, cand)

        rec_logits = model.rec_logits_from_candidates(enc_out["z_hat_q"], cand_proj)
        l_rec = F.cross_entropy(
            rec_logits,
            torch.zeros(rec_logits.size(0), dtype=torch.long, device=rec_logits.device)
        )

        loss = enc_losses["l_enc"] + LAMBDA_REC * l_rec

    stats = {
        "loss": loss.detach(),
        "l_enc": enc_losses["l_enc"].detach(),
        "l_pos": enc_losses["l_pos"].detach(),
        "l_next": enc_losses["l_next"].detach(),
        "l_task": enc_losses["l_task"].detach(),
        "l_rec": l_rec.detach(),
    }
    return loss, stats

def sum_step(batch, item_proj_table, sem_weight=SEM_LOSS_WEIGHT, freeze_encoder=False):
    task_ids = batch["task_id"]
    h_ids = batch["h_ids"]
    h_act = batch["h_act"]
    h_len = batch["h_len"]
    qid = batch["qid"]           # by construction qid == src_nid for summarization
    src_nid = batch["src_nid"]
    tgt_ids = batch["tgt_ids"]
    tgt_mask = batch["tgt_mask"]
    tgt_emb = batch["tgt_emb"]

    hist_item_emb = gather_hist_item_emb(h_ids)

    src_ids = SRC_IDS_T.index_select(0, src_nid.detach().cpu()).to(device, non_blocking=True)
    src_mask = SRC_MASK_T.index_select(0, src_nid.detach().cpu()).to(device, non_blocking=True)

    if freeze_encoder:
        with torch.no_grad():
            enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)
            e_dq = item_proj_table.index_select(0, src_nid).detach()
    else:
        enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)
        e_dq = item_proj_table.index_select(0, src_nid)

    with torch.cuda.amp.autocast(enabled=USE_AMP):
        if freeze_encoder:
            enc_losses = {
                "l_enc": torch.tensor(0.0, device=device),
                "l_pos": torch.tensor(0.0, device=device),
                "l_next": torch.tensor(0.0, device=device),
                "l_task": torch.tensor(0.0, device=device),
            }
        else:
            enc_losses = model.encoder.encoder_losses(enc_out, task_ids, qid, item_proj_table, h_len)

        l_sum, dec_stats = model.decoder.loss(
            z_hat_q=enc_out["z_hat_q"].detach() if freeze_encoder else enc_out["z_hat_q"],
            z_user=enc_out["z_last"].detach() if freeze_encoder else enc_out["z_last"],
            e_dq=e_dq,
            src_ids=src_ids,
            src_mask=src_mask,
            tgt_ids=tgt_ids,
            tgt_mask=tgt_mask,
            tgt_emb=tgt_emb,
            sem_weight=sem_weight,
        )

        loss = l_sum if freeze_encoder else (enc_losses["l_enc"] + LAMBDA_SUM * l_sum)

    stats = {
        "loss": loss.detach(),
        "l_enc": enc_losses["l_enc"].detach(),
        "l_pos": enc_losses["l_pos"].detach(),
        "l_next": enc_losses["l_next"].detach(),
        "l_task": enc_losses["l_task"].detach(),
        "l_sum": l_sum.detach(),
        "l_ce": dec_stats["l_ce"].detach(),
        "l_sem": dec_stats["l_sem"].detach(),
    }
    return loss, stats

print("Loss Functions and Metrics ready")

In [ ]:
# ============================================================
# CELL 11 — Validation functions and light summary metrics
# ============================================================
def ids_to_text(ids):
    return tokenizer.decode(
        [int(x) for x in ids if int(x) != tokenizer.pad_token_id],
        skip_special_tokens=True
    ).strip()

def lcs_len(a, b):
    n, m = len(a), len(b)
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        for j in range(1, m + 1):
            tmp = dp[j]
            if a[i - 1] == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]

def rouge_l_f1_text(hyp, ref):
    h = hyp.split()
    r = ref.split()
    if not h or not r:
        return 0.0
    l = lcs_len(h, r)
    p = l / max(1, len(h))
    q = l / max(1, len(r))
    return 0.0 if (p + q) == 0 else (2 * p * q) / (p + q)

def bleu1_text(hyp, ref):
    h = hyp.split()
    r = ref.split()
    if not h or not r:
        return 0.0
    hc = Counter(h)
    rc = Counter(r)
    overlap = sum((hc & rc).values())
    p = overlap / max(1, len(h))
    bp = 1.0 if len(h) > len(r) else math.exp(1 - len(r) / max(1, len(h)))
    return bp * p

def jsd_unigram_text(a, b, eps=1e-12):
    ta = a.split()
    tb = b.split()
    if len(ta) == 0 or len(tb) == 0:
        return 0.0

    va = Counter(ta)
    vb = Counter(tb)
    vocab = sorted(set(va) | set(vb))

    p = np.array([va[t] for t in vocab], dtype=np.float64)
    q = np.array([vb[t] for t in vocab], dtype=np.float64)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)

    def kl(x, y):
        x = np.clip(x, eps, 1.0)
        y = np.clip(y, eps, 1.0)
        return float(np.sum(x * np.log(x / y)))

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

@torch.no_grad()
def evaluate_rec(loader, item_proj_table, num_negs=REC_EVAL_NEGS, seed=9999):
    model.eval()
    gen = torch.Generator(device=device).manual_seed(seed)
    all_ranks = []

    for batch in tqdm(loader, desc="eval rec", leave=False):
        batch = to_device(batch)

        h_ids = batch["h_ids"]
        h_act = batch["h_act"]
        h_len = batch["h_len"]
        qid = batch["qid"]
        task_ids = batch["task_id"]

        hist_item_emb = gather_hist_item_emb(h_ids)
        enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)

        neg = sample_negs(qid, item_proj_table.size(0), num_negs, gen)
        cand = torch.cat([qid[:, None], neg], dim=1)
        cand_proj = gather_item_proj(item_proj_table, cand)
        logits = model.rec_logits_from_candidates(enc_out["z_hat_q"], cand_proj)

        rank = 1 + (logits[:, 1:] >= logits[:, :1]).sum(dim=1)
        all_ranks.append(rank.detach().cpu())

    ranks = torch.cat(all_ranks)
    return rec_metrics_from_ranks(ranks, Ks=(5, 10))

@torch.no_grad()
def evaluate_sum_loss(loader, item_proj_table, freeze_encoder=False, max_batches=None):
    model.eval()
    vals = []
    seen = 0

    for batch in tqdm(loader, desc="eval sum loss", leave=False):
        batch = to_device(batch)
        loss, _ = sum_step(batch, item_proj_table, sem_weight=SEM_LOSS_WEIGHT, freeze_encoder=freeze_encoder)
        vals.append(float(loss.detach().cpu()))
        seen += 1
        if max_batches is not None and seen >= max_batches:
            break

    return float(np.mean(vals)) if len(vals) > 0 else float("inf")

@torch.no_grad()
def evaluate_sum(loader, item_proj_table, max_batches=None):
    model.eval()
    rows = []
    seen_batches = 0

    for batch in tqdm(loader, desc="eval sum", leave=False):
        batch = to_device(batch)

        h_ids = batch["h_ids"]
        h_act = batch["h_act"]
        h_len = batch["h_len"]
        src_nid = batch["src_nid"]
        task_ids = batch["task_id"]

        hist_item_emb = gather_hist_item_emb(h_ids)
        enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)
        e_dq = item_proj_table.index_select(0, src_nid)

        src_ids = SRC_IDS_T.index_select(0, src_nid.detach().cpu()).to(device, non_blocking=True)
        src_mask = SRC_MASK_T.index_select(0, src_nid.detach().cpu()).to(device, non_blocking=True)

        gen_ids = model.decoder.generate(
            z_hat_q=enc_out["z_hat_q"],
            z_user=enc_out["z_last"],
            e_dq=e_dq,
            src_ids=src_ids,
            src_mask=src_mask,
            max_new_tokens=MAX_TGT_TOK - 1,
        )

        for i in range(gen_ids.size(0)):
            pred_txt = ids_to_text(gen_ids[i].tolist())
            gold_txt = batch["gold_text"][i]

            row = {
                "sid": batch["sid"][i],
                "userid": batch["userid"][i] if "userid" in batch else None,
                "gold_text": gold_txt,
                "pred_text": pred_txt,
                "ROUGE-L": rouge_l_f1_text(pred_txt, gold_txt),
                "BLEU-1": bleu1_text(pred_txt, gold_txt),
                "JSD": jsd_unigram_text(pred_txt, gold_txt),
            }
            rows.append(row)

        seen_batches += 1
        if max_batches is not None and seen_batches >= max_batches:
            break

    df = pd.DataFrame(rows)
    metrics = {
        "ROUGE-L": float(df["ROUGE-L"].mean()) if len(df) else 0.0,
        "BLEU-1": float(df["BLEU-1"].mean()) if len(df) else 0.0,
        "JSD": float(df["JSD"].mean()) if len(df) else 0.0,
    }
    return metrics, df

def stage2_joint_score(rec_metrics, sum_metrics):
    return (
        1.0 * rec_metrics["MRR"]
        + 1.0 * sum_metrics["ROUGE-L"]
        + 0.25 * sum_metrics["BLEU-1"]
        - 1.0 * sum_metrics["JSD"]
    )

print("Validation functions and light summary metrics ready")

In [ ]:
# ============================================================
# CELL 12 — Resume logic + exact 3-stage training loops
# ============================================================
def load_model_only(path):
    ckpt = torch.load(path, map_location=device)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    load_matching_state(model, state)
    return ckpt

def stage_resume_helper(last_ckpt_path, optimizer_builder, total_steps, warmup_ratio, bootstrap_path=None):
    optimizer = optimizer_builder()
    scheduler = build_warmup_cosine_scheduler(optimizer, total_steps, warmup_ratio)

    start_epoch = 1
    best_score = -1e18
    history_rows = []

    if os.path.exists(last_ckpt_path):
        ckpt = torch.load(last_ckpt_path, map_location=device)
        load_matching_state(model, ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if USE_AMP and ckpt.get("scaler") is not None:
            scaler.load_state_dict(ckpt["scaler"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_score = float(ckpt.get("best_score", best_score))
        history_rows = ckpt.get("history_rows", [])
        print("resumed:", last_ckpt_path, "next epoch:", start_epoch)
    else:
        if bootstrap_path is not None and os.path.exists(bootstrap_path):
            load_model_only(bootstrap_path)
            print("bootstrapped from:", bootstrap_path)

    return optimizer, scheduler, start_epoch, best_score, history_rows

def train_stage1_one_epoch(epoch_idx, optimizer, scheduler):
    configure_stage1_rec()
    item_proj_table = build_projected_item_table()
    rec_gen = torch.Generator(device=device).manual_seed(SEED + 1000 + epoch_idx)

    running = defaultdict(float)
    counts = defaultdict(int)

    pbar = tqdm(rec_train_loader, desc=f"stage1 rec epoch {epoch_idx:02d}", leave=True)
    for batch in pbar:
        batch = to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        loss, stats = rec_step(batch, item_proj_table, rec_gen)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running["loss"] += float(loss.detach().cpu())
        counts["loss"] += 1
        for k, v in stats.items():
            if k != "loss":
                running[k] += float(v.detach().cpu())
                counts[k] += 1

        pbar.set_postfix({
            "loss": f"{running['loss'] / max(1, counts['loss']):.4f}",
            "l_rec": f"{running['l_rec'] / max(1, counts['l_rec']):.4f}",
            "l_enc": f"{running['l_enc'] / max(1, counts['l_enc']):.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    return {k: running[k] / max(1, counts[k]) for k in running.keys()}

@torch.no_grad()
def validate_stage1(epoch_idx):
    model.eval()
    item_proj_table = build_projected_item_table()
    return evaluate_rec(rec_val_loader, item_proj_table, num_negs=REC_EVAL_NEGS, seed=epoch_idx + 5000)

def train_stage2_one_epoch(epoch_idx, optimizer, scheduler):
    configure_stage2_joint()
    item_proj_table = build_projected_item_table()

    rec_iter = iter(rec_train_loader)
    sum_iter = iter(sum_train_loader)
    max_steps = max(len(rec_train_loader), len(sum_train_loader))
    rec_gen = torch.Generator(device=device).manual_seed(SEED + 2000 + epoch_idx)

    running = defaultdict(float)
    counts = defaultdict(int)

    pbar = tqdm(range(max_steps), desc=f"stage2 joint epoch {epoch_idx:02d}", leave=True)
    for _ in pbar:
        optimizer.zero_grad(set_to_none=True)
        did_step = False

        try:
            rec_batch = next(rec_iter)
        except StopIteration:
            rec_batch = None

        if rec_batch is not None:
            rec_batch = to_device(rec_batch)
            loss_rec, stats_rec = rec_step(rec_batch, item_proj_table, rec_gen)
            scaler.scale(loss_rec / GRAD_ACCUM_STEPS).backward()
            did_step = True

            running["rec_total"] += float(loss_rec.detach().cpu())
            counts["rec_total"] += 1
            for k, v in stats_rec.items():
                if k != "loss":
                    running["rec_" + k] += float(v.detach().cpu())
                    counts["rec_" + k] += 1

        try:
            sum_batch = next(sum_iter)
        except StopIteration:
            sum_batch = None

        if sum_batch is not None:
            sum_batch = to_device(sum_batch)
            loss_sum, stats_sum = sum_step(
                sum_batch,
                item_proj_table,
                sem_weight=SEM_LOSS_WEIGHT,
                freeze_encoder=False,
            )
            scaler.scale(loss_sum / GRAD_ACCUM_STEPS).backward()
            did_step = True

            running["sum_total"] += float(loss_sum.detach().cpu())
            counts["sum_total"] += 1
            for k, v in stats_sum.items():
                if k != "loss":
                    running["sum_" + k] += float(v.detach().cpu())
                    counts["sum_" + k] += 1

        if did_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        pbar.set_postfix({
            "rec": f"{running['rec_total'] / max(1, counts['rec_total']):.4f}",
            "sum": f"{running['sum_total'] / max(1, counts['sum_total']):.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    return {k: running[k] / max(1, counts[k]) for k in running.keys()}

@torch.no_grad()
def validate_stage2(epoch_idx):
    model.eval()
    item_proj_table = build_projected_item_table()
    rec_metrics = evaluate_rec(rec_val_loader, item_proj_table, num_negs=REC_EVAL_NEGS, seed=epoch_idx + 9000)
    sum_metrics, _ = evaluate_sum(sum_val_loader, item_proj_table, max_batches=None)
    return rec_metrics, sum_metrics

def train_stage3_one_epoch(epoch_idx, optimizer, scheduler):
    configure_stage3_sumft()
    item_proj_table = build_projected_item_table()

    running = defaultdict(float)
    counts = defaultdict(int)

    pbar = tqdm(sum_train_loader, desc=f"stage3 sum-ft epoch {epoch_idx:02d}", leave=True)
    for batch in pbar:
        batch = to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        loss, stats = sum_step(
            batch,
            item_proj_table,
            sem_weight=SEM_LOSS_WEIGHT,
            freeze_encoder=True,
        )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running["loss"] += float(loss.detach().cpu())
        counts["loss"] += 1
        for k, v in stats.items():
            if k != "loss":
                running[k] += float(v.detach().cpu())
                counts[k] += 1

        pbar.set_postfix({
            "loss": f"{running['loss'] / max(1, counts['loss']):.4f}",
            "l_ce": f"{running['l_ce'] / max(1, counts['l_ce']):.4f}",
            "l_sem": f"{running['l_sem'] / max(1, counts['l_sem']):.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    return {k: running[k] / max(1, counts[k]) for k in running.keys()}

@torch.no_grad()
def validate_stage3(epoch_idx):
    model.eval()
    item_proj_table = build_projected_item_table()
    val_sum_loss = evaluate_sum_loss(sum_val_loader, item_proj_table, freeze_encoder=True, max_batches=None)
    sum_metrics, _ = evaluate_sum(sum_val_loader, item_proj_table, max_batches=None)
    return val_sum_loss, sum_metrics

print("Training Strategy ready")

In [ ]:
# ============================================================
# CELL 13 — Run Stage 1 -> Stage 2 -> Stage 3 with checkpoints and histories
# ============================================================
# ----------------------------
# STAGE 1 — recommendation only
# ----------------------------
s1_optimizer, s1_scheduler, s1_start_epoch, s1_best_score, s1_history = stage_resume_helper(
    STAGE1_LAST_CKPT,
    build_stage1_optimizer,
    total_steps=max(1, STAGE1_REC_EPOCHS * len(rec_train_loader)),
    warmup_ratio=STAGE1_WARMUP_RATIO,
    bootstrap_path=None,
)

for epoch in range(s1_start_epoch, STAGE1_REC_EPOCHS + 1):
    train_hist = train_stage1_one_epoch(epoch, s1_optimizer, s1_scheduler)
    rec_val_metrics = validate_stage1(epoch)
    score = float(rec_val_metrics["MRR"])

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_hist.items()},
        **{f"val_{k}": v for k, v in rec_val_metrics.items()},
        "score": score,
    }
    s1_history.append(row)
    pd.DataFrame(s1_history).to_csv(STAGE1_HIST_CSV, index=False)

    payload = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": s1_optimizer.state_dict(),
        "scheduler": s1_scheduler.state_dict(),
        "scaler": scaler.state_dict() if USE_AMP else None,
        "best_score": max(s1_best_score, score),
        "history_rows": s1_history,
        "stage": "stage1_rec_only",
    }
    save_ckpt(STAGE1_LAST_CKPT, payload)

    if score > s1_best_score:
        s1_best_score = score
        save_ckpt(STAGE1_BEST_CKPT, payload)

    print(f"[stage1 epoch {epoch:02d}] val_rec={rec_val_metrics} score={score:.6f}")

if os.path.exists(STAGE1_BEST_CKPT):
    load_model_only(STAGE1_BEST_CKPT)
    print("loaded best stage1 checkpoint:", STAGE1_BEST_CKPT)

# ----------------------------
# STAGE 2 — joint recommendation + summarization
# ----------------------------
s2_boot = STAGE1_BEST_CKPT if os.path.exists(STAGE1_BEST_CKPT) else None
s2_optimizer, s2_scheduler, s2_start_epoch, s2_best_score, s2_history = stage_resume_helper(
    STAGE2_LAST_CKPT,
    build_stage2_optimizer,
    total_steps=max(1, STAGE2_JOINT_EPOCHS * max(len(rec_train_loader), len(sum_train_loader))),
    warmup_ratio=STAGE2_WARMUP_RATIO,
    bootstrap_path=s2_boot,
)

for epoch in range(s2_start_epoch, STAGE2_JOINT_EPOCHS + 1):
    train_hist = train_stage2_one_epoch(epoch, s2_optimizer, s2_scheduler)
    rec_val_metrics, sum_val_metrics = validate_stage2(epoch)
    score = stage2_joint_score(rec_val_metrics, sum_val_metrics)

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_hist.items()},
        **{f"val_rec_{k}": v for k, v in rec_val_metrics.items()},
        **{f"val_sum_{k}": v for k, v in sum_val_metrics.items()},
        "joint_score": score,
    }
    s2_history.append(row)
    pd.DataFrame(s2_history).to_csv(STAGE2_HIST_CSV, index=False)

    payload = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": s2_optimizer.state_dict(),
        "scheduler": s2_scheduler.state_dict(),
        "scaler": scaler.state_dict() if USE_AMP else None,
        "best_score": max(s2_best_score, score),
        "history_rows": s2_history,
        "stage": "stage2_joint",
    }
    save_ckpt(STAGE2_LAST_CKPT, payload)

    if score > s2_best_score:
        s2_best_score = score
        save_ckpt(STAGE2_BEST_CKPT, payload)

    print(
        f"[stage2 epoch {epoch:02d}] "
        f"val_rec={rec_val_metrics} "
        f"val_sum={sum_val_metrics} "
        f"joint={score:.6f}"
    )

if os.path.exists(STAGE2_BEST_CKPT):
    load_model_only(STAGE2_BEST_CKPT)
    print("loaded best stage2 checkpoint:", STAGE2_BEST_CKPT)

# ----------------------------
# STAGE 3 — summary fine-tuning only
# ----------------------------
s3_boot = STAGE2_BEST_CKPT if os.path.exists(STAGE2_BEST_CKPT) else None
s3_optimizer, s3_scheduler, s3_start_epoch, s3_best_score, s3_history = stage_resume_helper(
    STAGE3_LAST_CKPT,
    build_stage3_optimizer,
    total_steps=max(1, STAGE3_SUM_FT_EPOCHS * len(sum_train_loader)),
    warmup_ratio=STAGE3_WARMUP_RATIO,
    bootstrap_path=s3_boot,
)

best_val_sum_loss = float("inf")
if len(s3_history) > 0 and "val_sum_loss" in s3_history[-1]:
    best_val_sum_loss = min(float(r["val_sum_loss"]) for r in s3_history)

for epoch in range(s3_start_epoch, STAGE3_SUM_FT_EPOCHS + 1):
    train_hist = train_stage3_one_epoch(epoch, s3_optimizer, s3_scheduler)
    val_sum_loss, sum_val_metrics = validate_stage3(epoch)
    score = -val_sum_loss

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_hist.items()},
        "val_sum_loss": val_sum_loss,
        **{f"val_sum_{k}": v for k, v in sum_val_metrics.items()},
        "score": score,
    }
    s3_history.append(row)
    pd.DataFrame(s3_history).to_csv(STAGE3_HIST_CSV, index=False)

    payload = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": s3_optimizer.state_dict(),
        "scheduler": s3_scheduler.state_dict(),
        "scaler": scaler.state_dict() if USE_AMP else None,
        "best_score": max(s3_best_score, score),
        "history_rows": s3_history,
        "best_val_sum_loss": min(best_val_sum_loss, val_sum_loss),
        "stage": "stage3_sum_ft",
    }
    save_ckpt(STAGE3_LAST_CKPT, payload)

    if val_sum_loss < best_val_sum_loss:
        best_val_sum_loss = val_sum_loss
        s3_best_score = score
        save_ckpt(STAGE3_BEST_CKPT, payload)

    print(
        f"[stage3 epoch {epoch:02d}] "
        f"val_sum_loss={val_sum_loss:.6f} "
        f"val_sum={sum_val_metrics}"
    )

print("\ntraining complete")
print("stage1 best:", STAGE1_BEST_CKPT if os.path.exists(STAGE1_BEST_CKPT) else "not created")
print("stage2 best:", STAGE2_BEST_CKPT if os.path.exists(STAGE2_BEST_CKPT) else "not created")
print("stage3 best:", STAGE3_BEST_CKPT if os.path.exists(STAGE3_BEST_CKPT) else "not created")

if len(s1_history) > 0:
    print("\nSTAGE1 history tail")
    display(pd.DataFrame(s1_history).tail())

if len(s2_history) > 0:
    print("\nSTAGE2 history tail")
    display(pd.DataFrame(s2_history).tail())

if len(s3_history) > 0:
    print("\nSTAGE3 history tail")
    display(pd.DataFrame(s3_history).tail())

In [ ]:
# ============================================================
# CELL 14 — Best-checkpoint validation summary and final model restore
# ============================================================
val_summary = {}

# -------------------------
# Stage 1 best checkpoint summary
# -------------------------
if os.path.exists(STAGE1_BEST_CKPT):
    load_model_only(STAGE1_BEST_CKPT)
    item_proj_table = build_projected_item_table()
    s1_val_rec = evaluate_rec(rec_val_loader, item_proj_table, num_negs=REC_EVAL_NEGS, seed=1111)
    val_summary["stage1_best_ckpt"] = STAGE1_BEST_CKPT
    val_summary["stage1_val_rec"] = s1_val_rec
    print("BEST STAGE1 VAL REC:", s1_val_rec)

# -------------------------
# Stage 2 best checkpoint summary
# -------------------------
if os.path.exists(STAGE2_BEST_CKPT):
    load_model_only(STAGE2_BEST_CKPT)
    item_proj_table = build_projected_item_table()
    s2_val_rec = evaluate_rec(rec_val_loader, item_proj_table, num_negs=REC_EVAL_NEGS, seed=2222)
    s2_val_sum, _ = evaluate_sum(sum_val_loader, item_proj_table, max_batches=None)
    s2_joint = stage2_joint_score(s2_val_rec, s2_val_sum)

    val_summary["stage2_best_ckpt"] = STAGE2_BEST_CKPT
    val_summary["stage2_val_rec"] = s2_val_rec
    val_summary["stage2_val_sum"] = s2_val_sum
    val_summary["stage2_joint_score"] = s2_joint

    print("BEST STAGE2 VAL REC:", s2_val_rec)
    print("BEST STAGE2 VAL SUM:", s2_val_sum)
    print("BEST STAGE2 JOINT  :", round(float(s2_joint), 6))

# -------------------------
# Stage 3 best checkpoint summary
# -------------------------
stage3_ckpt_to_use = None
if os.path.exists(STAGE3_BEST_CKPT):
    stage3_ckpt_to_use = STAGE3_BEST_CKPT
elif os.path.exists(STAGE3_LAST_CKPT):
    stage3_ckpt_to_use = STAGE3_LAST_CKPT
elif os.path.exists(STAGE2_BEST_CKPT):
    stage3_ckpt_to_use = STAGE2_BEST_CKPT

if stage3_ckpt_to_use is not None:
    load_model_only(stage3_ckpt_to_use)
    item_proj_table = build_projected_item_table()
    s3_val_sum_loss = evaluate_sum_loss(sum_val_loader, item_proj_table, freeze_encoder=True, max_batches=None)
    s3_val_sum, _ = evaluate_sum(sum_val_loader, item_proj_table, max_batches=None)

    val_summary["stage3_ckpt_used"] = stage3_ckpt_to_use
    val_summary["stage3_val_sum_loss"] = s3_val_sum_loss
    val_summary["stage3_val_sum"] = s3_val_sum

    print("STAGE3 CKPT USED   :", stage3_ckpt_to_use)
    print("STAGE3 VAL LOSS    :", round(float(s3_val_sum_loss), 6))
    print("STAGE3 VAL SUM     :", s3_val_sum)

# -------------------------
# Save validation summary
# -------------------------
with open(os.path.join(EVAL_DIR, "validation_summary_stage1_2_3.json"), "w") as f:
    json.dump(val_summary, f, indent=2)

print("\nsaved:", os.path.join(EVAL_DIR, "validation_summary_stage1_2_3.json"))

# -------------------------
# Restore final model for downstream evaluation:
# prefer stage3 best, else stage2 best
# -------------------------
final_ckpt = None
if os.path.exists(STAGE3_BEST_CKPT):
    final_ckpt = STAGE3_BEST_CKPT
elif os.path.exists(STAGE2_BEST_CKPT):
    final_ckpt = STAGE2_BEST_CKPT
elif os.path.exists(STAGE1_BEST_CKPT):
    final_ckpt = STAGE1_BEST_CKPT

if final_ckpt is not None:
    load_model_only(final_ckpt)
    print("final model restored from:", final_ckpt)
else:
    print("no checkpoint found to restore")

In [ ]:
# ============================================================
# EXTERNAL RECO-ONLY EVAL USING STAGE-1 RECOMMENDATION CHECKPOINT
# No training. No joint checkpoint. No summarization generation.
# ============================================================

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

PRED_DIR = os.path.join(EVAL_DIR, "stored_predictions")
os.makedirs(PRED_DIR, exist_ok=True)

REC_EXT_STAGE1_PRED_CSV = os.path.join(
    PRED_DIR,
    "external_test_reco_predictions_stage1_rec_only.csv"
)

REC_EXT_STAGE1_METRIC_CSV = os.path.join(
    EVAL_DIR,
    "external_test_reco_metrics_stage1_rec_only.csv"
)

REC_EXT_STAGE1_METRIC_JSON = os.path.join(
    EVAL_DIR,
    "external_test_reco_metrics_stage1_rec_only.json"
)

if not os.path.exists(STAGE1_BEST_CKPT):
    raise FileNotFoundError(
        f"Recommendation-only checkpoint not found: {STAGE1_BEST_CKPT}\n"
        "You need to rerun Stage 1 only. Do not rerun Stage 2/Stage 3 for reco-only external eval."
    )

if ext_loader is None:
    raise RuntimeError(
        "ext_loader is None. External data was not loaded/rebuilt correctly. "
        "Run the setup/data-loader cells first."
    )

print("Loading recommendation-only checkpoint:")
print(STAGE1_BEST_CKPT)

ckpt_obj = torch.load(STAGE1_BEST_CKPT, map_location=device)
print("checkpoint stage:", ckpt_obj.get("stage", "unknown") if isinstance(ckpt_obj, dict) else "raw_state_dict")

load_model_only(STAGE1_BEST_CKPT)
model.eval()

item_proj_table = build_projected_item_table()


def _safe_meta_slice(meta_df, start_idx, bs):
    if meta_df is None:
        return None
    return meta_df.iloc[start_idx:start_idx + bs].reset_index(drop=True)


@torch.no_grad()
def generate_external_reco_predictions_stage1(
    loader,
    out_csv,
    item_proj_table,
    num_negs=REC_EVAL_NEGS,
    seed=12345,
    meta_df=None,
):
    model.eval()
    gen = torch.Generator(device=device).manual_seed(seed)

    rows = []
    global_idx = 0

    for batch in tqdm(loader, desc="external reco predictions with stage1 checkpoint", leave=False):
        batch = to_device(batch)

        h_ids = batch["h_ids"]
        h_act = batch["h_act"]
        h_len = batch["h_len"]
        qid   = batch["qid"]
        bs = h_ids.size(0)

        # CRITICAL: external loader comes from ExtDataset, whose task_id is "sum".
        # For recommendation-only evaluation, force task_id to "rec".
        task_ids = torch.full_like(
            h_len,
            int(TASK2ID["rec"]),
            dtype=torch.long,
            device=device
        )

        hist_item_emb = gather_hist_item_emb(h_ids)
        enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)

        neg = sample_negs(qid, item_proj_table.size(0), num_negs, gen)
        cand = torch.cat([qid[:, None], neg], dim=1)
        cand_proj = gather_item_proj(item_proj_table, cand)

        logits = model.rec_logits_from_candidates(enc_out["z_hat_q"], cand_proj)
        rank = 1 + (logits[:, 1:] >= logits[:, :1]).sum(dim=1)

        cand_cpu = cand.detach().cpu().numpy()
        logits_cpu = logits.detach().cpu().numpy()
        rank_cpu = rank.detach().cpu().numpy()
        qid_cpu = qid.detach().cpu().numpy()
        hlen_cpu = h_len.detach().cpu().numpy()

        meta_slice = _safe_meta_slice(meta_df, global_idx, bs)

        for i in range(bs):
            order = np.argsort(-logits_cpu[i])
            top_ids = cand_cpu[i][order[:10]].tolist()
            top_scores = logits_cpu[i][order[:10]].astype(float).tolist()

            row = {
                "example_idx": int(global_idx + i),
                "qid": int(qid_cpu[i]),
                "h_len": int(hlen_cpu[i]),
                "rank": int(rank_cpu[i]),
                "pos_score": float(logits_cpu[i][0]),
                "cand_ids_json": json.dumps(cand_cpu[i].astype(int).tolist()),
                "cand_scores_json": json.dumps(logits_cpu[i].astype(float).tolist()),
                "top10_ids_json": json.dumps(top_ids),
                "top10_scores_json": json.dumps(top_scores),
                "checkpoint_used": STAGE1_BEST_CKPT,
                "checkpoint_type": "stage1_rec_only",
                "forced_task_id": "rec",
            }

            if "userid" in batch:
                row["userid"] = str(batch["userid"][i])
            if "sid" in batch:
                row["sid"] = str(batch["sid"][i])
            if "src_nid" in batch:
                row["src_nid"] = int(batch["src_nid"][i].detach().cpu().item())

            if meta_slice is not None:
                for c in meta_slice.columns:
                    if c in {"h_ids", "h_act"}:
                        continue
                    if c not in row:
                        v = meta_slice.loc[i, c]
                        if isinstance(v, np.ndarray):
                            continue
                        row[c] = v

            rows.append(row)

        global_idx += bs

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print("saved external reco predictions:", out_csv)
    print("shape:", df.shape)
    return df


def parse_json_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return json.loads(x)


def auc_from_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    pos = scores[0]
    neg = scores[1:]
    if len(neg) == 0:
        return 1.0
    gt = float(np.sum(pos > neg))
    eq = float(np.sum(pos == neg))
    return float((gt + 0.5 * eq) / len(neg))


def swerr_from_arrays(cand_ids, cand_scores, item_emb_table, k=5):
    cand_ids_t = torch.tensor(cand_ids, dtype=torch.long, device=device).unsqueeze(0)
    logits_t = torch.tensor(cand_scores, dtype=torch.float32, device=device).unsqueeze(0)

    order = torch.argsort(logits_t, dim=1, descending=True)
    topk_order = order[:, :min(k, cand_ids_t.size(1))]
    ranked_ids = cand_ids_t.gather(1, topk_order)

    flat_ranked = ranked_ids.reshape(-1)
    ranked_emb = item_emb_table.index_select(0, flat_ranked).view(1, -1, item_emb_table.size(1))

    gold_ids = cand_ids_t[:, 0]
    gold_emb = item_emb_table.index_select(0, gold_ids).unsqueeze(1)

    rel = F.cosine_similarity(
        ranked_emb,
        gold_emb.expand_as(ranked_emb),
        dim=-1
    ).clamp(-1.0, 1.0)

    rel = 0.5 * (rel + 1.0)

    not_sat = torch.ones(1, device=device)
    swerr = torch.zeros(1, device=device)

    for r in range(rel.size(1)):
        rr = rel[:, r].clamp(0.0, 1.0)
        swerr = swerr + not_sat * rr / float(r + 1)
        not_sat = not_sat * (1.0 - rr)

    return float(swerr.item())


def reco_metrics_from_stored_df(df, name):
    item_emb_sem = ITEM_EMB_CPU.to(device)

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"parse reco rows {name}", leave=False):
        rank = int(r["rank"])
        cand_ids = parse_json_list(r["cand_ids_json"])
        cand_scores = parse_json_list(r["cand_scores_json"])

        rows.append({
            "rank": rank,
            "auc": auc_from_scores(cand_scores),
            "swerr5": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=5),
            "swerr10": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=10),
            "swerr20": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=20),
        })

    tmp = pd.DataFrame(rows)
    ranks = tmp["rank"].to_numpy(dtype=np.float64)

    metrics = {
        "MRR": float(np.mean(1.0 / ranks)),
        "Hit@1": float(np.mean(ranks <= 1)),
        "Hit@3": float(np.mean(ranks <= 3)),
        "Hit@5": float(np.mean(ranks <= 5)),
        "Hit@10": float(np.mean(ranks <= 10)),
        "Recall@1": float(np.mean(ranks <= 1)),
        "Recall@3": float(np.mean(ranks <= 3)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "Recall@10": float(np.mean(ranks <= 10)),
        "Precision@1": float(np.mean(ranks <= 1) / 1.0),
        "Precision@3": float(np.mean(ranks <= 3) / 3.0),
        "Precision@5": float(np.mean(ranks <= 5) / 5.0),
        "Precision@10": float(np.mean(ranks <= 10) / 10.0),
        "NDCG@3": float(np.mean((ranks <= 3) * (1.0 / np.log2(ranks + 1.0)))),
        "NDCG@5": float(np.mean((ranks <= 5) * (1.0 / np.log2(ranks + 1.0)))),
        "NDCG@10": float(np.mean((ranks <= 10) * (1.0 / np.log2(ranks + 1.0)))),
        "AUC": float(tmp["auc"].mean()),
        "SWERR@5": float(tmp["swerr5"].mean()),
        "SWERR@10": float(tmp["swerr10"].mean()),
        "SWERR@20": float(tmp["swerr20"].mean()),
        "InvMeanRank": float(1.0 / np.mean(ranks)),
    }

    return metrics


rec_ext_stage1_pred_df = generate_external_reco_predictions_stage1(
    loader=ext_loader,
    out_csv=REC_EXT_STAGE1_PRED_CSV,
    item_proj_table=item_proj_table,
    num_negs=REC_EVAL_NEGS,
    seed=12345,
    meta_df=ext_eval_df.reset_index(drop=True),
)

external_reco_stage1_metrics = reco_metrics_from_stored_df(
    rec_ext_stage1_pred_df,
    "external_test_stage1_rec_only"
)

metric_row = {
    "dataset": "external_test",
    "checkpoint_type": "stage1_rec_only",
    "checkpoint": STAGE1_BEST_CKPT,
    **external_reco_stage1_metrics,
}

metric_df = pd.DataFrame([metric_row])
metric_df.to_csv(REC_EXT_STAGE1_METRIC_CSV, index=False)

with open(REC_EXT_STAGE1_METRIC_JSON, "w") as f:
    json.dump([metric_row], f, indent=2)

print("EXTERNAL RECO METRICS — STAGE1 RECO-ONLY CHECKPOINT")
display(metric_df.T.rename(columns={0: "value"}))

print("saved:", REC_EXT_STAGE1_METRIC_CSV)
print("saved:", REC_EXT_STAGE1_METRIC_JSON)

In [ ]:
# # ============================================================
# # CELL 15 — Generate and store predictions for BOTH tasks
# # internal test + external test
# # ============================================================
# import os, json, math, re
# import numpy as np
# import pandas as pd
# from tqdm.auto import tqdm

# PRED_DIR = os.path.join(EVAL_DIR, "stored_predictions")
# os.makedirs(PRED_DIR, exist_ok=True)

# REC_TEST_PRED_CSV = os.path.join(PRED_DIR, "internal_test_reco_predictions.csv")
# REC_EXT_PRED_CSV  = os.path.join(PRED_DIR, "external_test_reco_predictions.csv")
# SUM_TEST_PRED_CSV = os.path.join(PRED_DIR, "internal_test_sum_predictions.csv")
# SUM_EXT_PRED_CSV  = os.path.join(PRED_DIR, "external_test_sum_predictions.csv")
# PRED_MANIFEST_JSON = os.path.join(PRED_DIR, "prediction_manifest.json")

# def _safe_meta_slice(meta_df, start_idx, bs):
#     if meta_df is None:
#         return None
#     return meta_df.iloc[start_idx:start_idx + bs].reset_index(drop=True)

# @torch.no_grad()
# def generate_reco_predictions_to_csv(
#     loader,
#     out_csv,
#     item_proj_table,
#     num_negs,
#     seed,
#     forced_task_id=None,
#     meta_df=None,
# ):
#     model.eval()
#     gen = torch.Generator(device=device).manual_seed(seed)

#     rows = []
#     global_idx = 0

#     for batch in tqdm(loader, desc=f"reco predictions -> {os.path.basename(out_csv)}", leave=False):
#         batch = to_device(batch)

#         h_ids = batch["h_ids"]
#         h_act = batch["h_act"]
#         h_len = batch["h_len"]
#         qid   = batch["qid"]
#         bs = h_ids.size(0)

#         if forced_task_id is None:
#             task_ids = batch["task_id"]
#             if not torch.is_tensor(task_ids):
#                 task_ids = torch.tensor(task_ids, dtype=torch.long, device=device)
#         else:
#             task_ids = torch.full_like(h_len, int(forced_task_id), dtype=torch.long, device=device)

#         hist_item_emb = gather_hist_item_emb(h_ids)
#         enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)

#         neg = sample_negs(qid, item_proj_table.size(0), num_negs, gen)
#         cand = torch.cat([qid[:, None], neg], dim=1)
#         cand_proj = gather_item_proj(item_proj_table, cand)
#         logits = model.rec_logits_from_candidates(enc_out["z_hat_q"], cand_proj)

#         rank = 1 + (logits[:, 1:] >= logits[:, :1]).sum(dim=1)

#         cand_cpu = cand.detach().cpu().numpy()
#         logits_cpu = logits.detach().cpu().numpy()
#         rank_cpu = rank.detach().cpu().numpy()
#         qid_cpu = qid.detach().cpu().numpy()
#         hlen_cpu = h_len.detach().cpu().numpy()

#         meta_slice = _safe_meta_slice(meta_df, global_idx, bs)

#         for i in range(bs):
#             order = np.argsort(-logits_cpu[i])
#             top_ids = cand_cpu[i][order[:10]].tolist()
#             top_scores = logits_cpu[i][order[:10]].astype(float).tolist()

#             row = {
#                 "example_idx": int(global_idx + i),
#                 "qid": int(qid_cpu[i]),
#                 "h_len": int(hlen_cpu[i]),
#                 "rank": int(rank_cpu[i]),
#                 "pos_score": float(logits_cpu[i][0]),
#                 "cand_ids_json": json.dumps(cand_cpu[i].astype(int).tolist()),
#                 "cand_scores_json": json.dumps(logits_cpu[i].astype(float).tolist()),
#                 "top10_ids_json": json.dumps(top_ids),
#                 "top10_scores_json": json.dumps(top_scores),
#             }

#             if meta_slice is not None:
#                 for c in meta_slice.columns:
#                     if c in {"h_ids", "h_act"}:
#                         continue
#                     v = meta_slice.loc[i, c]
#                     if isinstance(v, np.ndarray):
#                         continue
#                     row[c] = v

#             rows.append(row)

#         global_idx += bs

#     df = pd.DataFrame(rows)
#     df.to_csv(out_csv, index=False)
#     print("saved reco predictions:", out_csv, "shape=", df.shape)
#     return df

# @torch.no_grad()
# def generate_sum_predictions_to_csv(
#     loader,
#     out_csv,
#     item_proj_table,
#     meta_df=None,
# ):
#     model.eval()

#     rows = []
#     global_idx = 0

#     for batch in tqdm(loader, desc=f"sum predictions -> {os.path.basename(out_csv)}", leave=False):
#         batch = to_device(batch)

#         h_ids = batch["h_ids"]
#         h_act = batch["h_act"]
#         h_len = batch["h_len"]
#         src_nid = batch["src_nid"]
#         qid = batch["qid"]
#         task_ids = batch["task_id"]
#         bs = h_ids.size(0)

#         hist_item_emb = gather_hist_item_emb(h_ids)
#         enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)
#         e_dq = item_proj_table.index_select(0, src_nid)

#         src_ids = SRC_IDS_T.index_select(0, src_nid.detach().cpu()).to(device, non_blocking=True)
#         src_mask = SRC_MASK_T.index_select(0, src_nid.detach().cpu()).to(device, non_blocking=True)

#         gen_ids = model.decoder.generate(
#             z_hat_q=enc_out["z_hat_q"],
#             z_user=enc_out["z_last"],
#             e_dq=e_dq,
#             src_ids=src_ids,
#             src_mask=src_mask,
#             max_new_tokens=MAX_TGT_TOK - 1,
#         )

#         src_nid_cpu = src_nid.detach().cpu().numpy()
#         qid_cpu = qid.detach().cpu().numpy()
#         hlen_cpu = h_len.detach().cpu().numpy()

#         meta_slice = _safe_meta_slice(meta_df, global_idx, bs)

#         for i in range(bs):
#             sid_i = batch["sid"][i]
#             pred_txt = ids_to_text(gen_ids[i].detach().cpu().tolist())
#             gold_txt = batch["gold_text"][i]
#             src_i = int(src_nid_cpu[i])

#             row = {
#                 "example_idx": int(global_idx + i),
#                 "sid": str(sid_i),
#                 "qid": int(qid_cpu[i]),
#                 "src_nid": src_i,
#                 "h_len": int(hlen_cpu[i]),
#                 "source_text": source_text_arr[src_i] if 0 <= src_i < len(source_text_arr) else "",
#                 "gold_text": str(gold_txt),
#                 "pred_text": str(pred_txt),
#             }

#             if "userid" in batch:
#                 row["userid"] = str(batch["userid"][i])

#             if meta_slice is not None:
#                 for c in meta_slice.columns:
#                     if c in {"h_ids", "h_act", "gold_text", "source_text", "pred_text"}:
#                         continue
#                     if c not in row:
#                         v = meta_slice.loc[i, c]
#                         if isinstance(v, np.ndarray):
#                             continue
#                         row[c] = v

#             rows.append(row)

#         global_idx += bs

#     df = pd.DataFrame(rows)

#     # ensure internal test also gets userid if available in meta
#     if "userid" not in df.columns and meta_df is not None and "userid" in meta_df.columns:
#         df = df.merge(
#             meta_df[["sid", "userid"]].drop_duplicates(subset=["sid"]),
#             on="sid",
#             how="left"
#         )

#     df.to_csv(out_csv, index=False)
#     print("saved summarization predictions:", out_csv, "shape=", df.shape)
#     return df

# # -------------------------
# # checkpoint selection
# # -------------------------
# rec_ckpt = STAGE2_BEST_CKPT if os.path.exists(STAGE2_BEST_CKPT) else STAGE2_LAST_CKPT
# sum_ckpt = STAGE3_BEST_CKPT if os.path.exists(STAGE3_BEST_CKPT) else (
#     STAGE3_LAST_CKPT if os.path.exists(STAGE3_LAST_CKPT) else rec_ckpt
# )

# if not os.path.exists(rec_ckpt):
#     raise FileNotFoundError(rec_ckpt)
# if not os.path.exists(sum_ckpt):
#     raise FileNotFoundError(sum_ckpt)

# # -------------------------
# # internal test reco
# # -------------------------
# load_model_only(rec_ckpt)
# item_proj_table = build_projected_item_table()
# rec_test_pred_df = generate_reco_predictions_to_csv(
#     loader=rec_test_loader,
#     out_csv=REC_TEST_PRED_CSV,
#     item_proj_table=item_proj_table,
#     num_negs=REC_EVAL_NEGS,
#     seed=9999,
#     forced_task_id=None,
#     meta_df=rec_te.reset_index(drop=True),
# )

# # -------------------------
# # external test reco
# # -------------------------
# if ext_loader is None:
#     rec_ext_pred_df = None
#     print("external loader missing: skipping external reco predictions")
# else:
#     load_model_only(rec_ckpt)
#     item_proj_table = build_projected_item_table()
#     rec_ext_pred_df = generate_reco_predictions_to_csv(
#         loader=ext_loader,
#         out_csv=REC_EXT_PRED_CSV,
#         item_proj_table=item_proj_table,
#         num_negs=REC_EVAL_NEGS,
#         seed=12345,
#         forced_task_id=TASK2ID["rec"],
#         meta_df=ext_eval_df.reset_index(drop=True),
#     )

# # -------------------------
# # internal test summarization
# # -------------------------
# load_model_only(sum_ckpt)
# item_proj_table = build_projected_item_table()
# sum_test_pred_df = generate_sum_predictions_to_csv(
#     loader=sum_test_loader,
#     out_csv=SUM_TEST_PRED_CSV,
#     item_proj_table=item_proj_table,
#     meta_df=sum_te.reset_index(drop=True),
# )

# # -------------------------
# # external test summarization
# # -------------------------
# if ext_loader is None:
#     sum_ext_pred_df = None
#     print("external loader missing: skipping external summarization predictions")
# else:
#     load_model_only(sum_ckpt)
#     item_proj_table = build_projected_item_table()
#     sum_ext_pred_df = generate_sum_predictions_to_csv(
#         loader=ext_loader,
#         out_csv=SUM_EXT_PRED_CSV,
#         item_proj_table=item_proj_table,
#         meta_df=ext_eval_df.reset_index(drop=True),
#     )

# manifest = {
#     "rec_ckpt": rec_ckpt,
#     "sum_ckpt": sum_ckpt,
#     "internal_test_reco_csv": REC_TEST_PRED_CSV,
#     "external_test_reco_csv": REC_EXT_PRED_CSV if ext_loader is not None else None,
#     "internal_test_sum_csv": SUM_TEST_PRED_CSV,
#     "external_test_sum_csv": SUM_EXT_PRED_CSV if ext_loader is not None else None,
# }
# with open(PRED_MANIFEST_JSON, "w") as f:
#     json.dump(manifest, f, indent=2)

# print("saved manifest:", PRED_MANIFEST_JSON)

In [ ]:
# ============================================================
# CELL 16 — Recommendation metrics from STORED prediction CSVs
# all metrics higher-is-better
# internal test + external test
# ============================================================
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

PRED_DIR = os.path.join(EVAL_DIR, "stored_predictions")
os.makedirs(PRED_DIR, exist_ok=True)
REC_TEST_PRED_CSV = os.path.join(PRED_DIR, "internal_test_reco_predictions.csv")
REC_EXT_PRED_CSV  = os.path.join(PRED_DIR, "external_test_reco_predictions.csv")
SUM_TEST_PRED_CSV = os.path.join(PRED_DIR, "internal_test_sum_predictions.csv")
SUM_EXT_PRED_CSV  = os.path.join(PRED_DIR, "external_test_sum_predictions.csv")
PRED_MANIFEST_JSON = os.path.join(PRED_DIR, "prediction_manifest.json")

RECO_METRIC_JSON = os.path.join(EVAL_DIR, "reco_metrics_from_stored_predictions.json")
RECO_METRIC_CSV  = os.path.join(EVAL_DIR, "reco_metrics_from_stored_predictions.csv")

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return json.loads(x)

def swerr_from_arrays(cand_ids, cand_scores, item_emb_table, k=5):
    cand_ids_t = torch.tensor(cand_ids, dtype=torch.long, device=device).unsqueeze(0)
    logits_t   = torch.tensor(cand_scores, dtype=torch.float32, device=device).unsqueeze(0)

    order = torch.argsort(logits_t, dim=1, descending=True)
    topk_order = order[:, :min(k, cand_ids_t.size(1))]
    ranked_ids = cand_ids_t.gather(1, topk_order)

    flat_ranked = ranked_ids.reshape(-1)
    ranked_emb = item_emb_table.index_select(0, flat_ranked).view(1, -1, item_emb_table.size(1))

    gold_ids = cand_ids_t[:, 0]
    gold_emb = item_emb_table.index_select(0, gold_ids).unsqueeze(1)

    rel = F.cosine_similarity(ranked_emb, gold_emb.expand_as(ranked_emb), dim=-1).clamp(-1.0, 1.0)
    rel = 0.5 * (rel + 1.0)

    not_sat = torch.ones(1, device=device)
    swerr = torch.zeros(1, device=device)

    for r in range(rel.size(1)):
        rr = rel[:, r].clamp(0.0, 1.0)
        swerr = swerr + not_sat * rr / float(r + 1)
        not_sat = not_sat * (1.0 - rr)

    return float(swerr.item())

def auc_from_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    pos = scores[0]
    neg = scores[1:]
    if len(neg) == 0:
        return 1.0
    gt = float(np.sum(pos > neg))
    eq = float(np.sum(pos == neg))
    return float((gt + 0.5 * eq) / len(neg))

def reco_metrics_from_stored_df(df, name):
    item_emb_sem = ITEM_EMB_CPU.to(device)

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"parse reco rows {name}", leave=False):
        rank = int(r["rank"])
        cand_ids = parse_json_list(r["cand_ids_json"])
        cand_scores = parse_json_list(r["cand_scores_json"])

        row = {
            "rank": rank,
            "auc": auc_from_scores(cand_scores),
            "swerr5": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=5),
            "swerr10": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=10),
            "swerr20": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=20),
        }
        rows.append(row)

    tmp = pd.DataFrame(rows)
    ranks = tmp["rank"].to_numpy(dtype=np.float64)

    metrics = {
        "MRR": float(np.mean(1.0 / ranks)),
        "Hit@1": float(np.mean(ranks <= 1)),
        "Hit@3": float(np.mean(ranks <= 3)),
        "Hit@5": float(np.mean(ranks <= 5)),
        "Hit@10": float(np.mean(ranks <= 10)),
        "Recall@1": float(np.mean(ranks <= 1)),
        "Recall@3": float(np.mean(ranks <= 3)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "Recall@10": float(np.mean(ranks <= 10)),
        "Precision@1": float(np.mean(ranks <= 1) / 1.0),
        "Precision@3": float(np.mean(ranks <= 3) / 3.0),
        "Precision@5": float(np.mean(ranks <= 5) / 5.0),
        "Precision@10": float(np.mean(ranks <= 10) / 10.0),
        "NDCG@3": float(np.mean((ranks <= 3) * (1.0 / np.log2(ranks + 1.0)))),
        "NDCG@5": float(np.mean((ranks <= 5) * (1.0 / np.log2(ranks + 1.0)))),
        "NDCG@10": float(np.mean((ranks <= 10) * (1.0 / np.log2(ranks + 1.0)))),
        "AUC": float(tmp["auc"].mean()),
        "SWERR@5": float(tmp["swerr5"].mean()),
        "SWERR@10": float(tmp["swerr10"].mean()),
        "SWERR@20": float(tmp["swerr20"].mean()),
        "InvMeanRank": float(1.0 / np.mean(ranks)),
    }
    return metrics

rec_test_pred_df = pd.read_csv(REC_TEST_PRED_CSV)
internal_reco_metrics = reco_metrics_from_stored_df(rec_test_pred_df, "internal_test")

results = [{"dataset": "internal_test", **internal_reco_metrics}]
print("INTERNAL TEST RECO METRICS")
display(pd.DataFrame([internal_reco_metrics]).T.rename(columns={0: "value"}))

if os.path.exists(REC_EXT_PRED_CSV):
    rec_ext_pred_df = pd.read_csv(REC_EXT_PRED_CSV)
    external_reco_metrics = reco_metrics_from_stored_df(rec_ext_pred_df, "external_test")
    results.append({"dataset": "external_test", **external_reco_metrics})

    print("EXTERNAL TEST RECO METRICS")
    display(pd.DataFrame([external_reco_metrics]).T.rename(columns={0: "value"}))
else:
    external_reco_metrics = None
    print("external reco prediction csv missing; skipping")

pd.DataFrame(results).to_csv(RECO_METRIC_CSV, index=False)
with open(RECO_METRIC_JSON, "w") as f:
    json.dump(results, f, indent=2)

print("saved:", RECO_METRIC_CSV)
print("saved:", RECO_METRIC_JSON)

In [ ]:
# ============================================================
# CELL 17A — Lexical and traditional summary metrics
# raw overlap metrics only
# updates shared json/csv after completion
# ============================================================
import os
import re
import math
import json
import numpy as np
import pandas as pd
from collections import Counter
from functools import lru_cache
from tqdm.auto import tqdm

# -------------------------
# Shared paths
# -------------------------
SUM_METRIC_JSON = os.path.join(EVAL_DIR, "summary_accuracy_metrics_from_stored_predictions.json")
SUM_METRIC_CSV  = os.path.join(EVAL_DIR, "summary_accuracy_metrics_from_stored_predictions.csv")

SUM_TEST_ENRICHED_CSV = os.path.join(PRED_DIR, "internal_test_sum_predictions_enriched.csv")
SUM_EXT_ENRICHED_CSV  = os.path.join(PRED_DIR, "external_test_sum_predictions_enriched.csv")

_TOKEN_RE = re.compile(r"[A-Za-z0-9']+")

# -------------------------
# Shared helpers
# -------------------------
def _clip01(x):
    if pd.isna(x):
        return np.nan
    return float(max(0.0, min(1.0, float(x))))

def _normalize_metric_text(x):
    if x is None or pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

@lru_cache(maxsize=500000)
def tok_text_cached(text):
    return tuple(_TOKEN_RE.findall(str(text).lower()))

def _ngrams(tokens, n):
    L = len(tokens)
    if L < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(L - n + 1)]

def rouge_n_f1_tokens(pred_toks, gold_toks, n=1):
    p = Counter(_ngrams(pred_toks, n))
    g = Counter(_ngrams(gold_toks, n))
    if len(p) == 0 or len(g) == 0:
        return 0.0
    overlap = sum((p & g).values())
    prec = overlap / max(1, sum(p.values()))
    rec  = overlap / max(1, sum(g.values()))
    return 0.0 if (prec + rec) == 0 else (2.0 * prec * rec / (prec + rec))

def lcs_len(a, b):
    n, m = len(a), len(b)
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        ai = a[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            if ai == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]

def rouge_l_f1_tokens(pred_toks, gold_toks):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    l = lcs_len(pred_toks, gold_toks)
    p = l / max(1, len(pred_toks))
    r = l / max(1, len(gold_toks))
    return 0.0 if (p + r) == 0 else (2.0 * p * r / (p + r))

def _skip_bigrams_plus_unigrams(tokens, max_skip=4):
    out = []
    L = len(tokens)
    for i in range(L):
        out.append((tokens[i],))
        hi = min(L, i + max_skip + 2)
        for j in range(i + 1, hi):
            out.append((tokens[i], tokens[j]))
    return out

def rouge_su4_f1_tokens(pred_toks, gold_toks, max_skip=4):
    p = Counter(_skip_bigrams_plus_unigrams(pred_toks, max_skip=max_skip))
    g = Counter(_skip_bigrams_plus_unigrams(gold_toks, max_skip=max_skip))
    if len(p) == 0 or len(g) == 0:
        return 0.0
    overlap = sum((p & g).values())
    prec = overlap / max(1, sum(p.values()))
    rec  = overlap / max(1, sum(g.values()))
    return 0.0 if (prec + rec) == 0 else (2.0 * prec * rec / (prec + rec))

def meteor_tokens(pred_toks, gold_toks):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    from nltk.translate.meteor_score import meteor_score
    return float(meteor_score([list(gold_toks)], list(pred_toks)))

def bleu_score_tokens(pred_toks, gold_toks, max_n=4):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    log_prec_sum = 0.0
    for n in range(1, max_n + 1):
        p = Counter(_ngrams(pred_toks, n))
        g = Counter(_ngrams(gold_toks, n))
        if len(p) == 0:
            return 0.0
        overlap = sum((p & g).values())
        total = sum(p.values())
        prec_n = (overlap + 1.0) / (total + 1.0)
        log_prec_sum += (1.0 / max_n) * math.log(max(prec_n, 1e-12))
    c = len(pred_toks)
    r = len(gold_toks)
    bp = 1.0 if c > r else math.exp(1.0 - (r / max(c, 1)))
    return float(bp * math.exp(log_prec_sum))

def jsd_dist_tokens(pred_toks, gold_toks):
    vocab = sorted(set(pred_toks) | set(gold_toks))
    if len(vocab) == 0:
        return 0.0
    pc = Counter(pred_toks)
    gc = Counter(gold_toks)
    p = np.asarray([pc[t] for t in vocab], dtype=np.float64)
    q = np.asarray([gc[t] for t in vocab], dtype=np.float64)
    p = p / max(1e-12, p.sum())
    q = q / max(1e-12, q.sum())
    m = 0.5 * (p + q)

    def _kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / np.maximum(b[mask], 1e-12)))

    return float(np.clip(0.5 * _kl(p, m) + 0.5 * _kl(q, m), 0.0, 1.0))

def traditional_metric_row(pred_text, gold_text):
    pred_text = _normalize_metric_text(pred_text)
    gold_text = _normalize_metric_text(gold_text)

    pt = tok_text_cached(pred_text)
    gt = tok_text_cached(gold_text)

    out = {}
    out["ROUGE-1"] = _clip01(rouge_n_f1_tokens(pt, gt, 1))
    out["ROUGE-2"] = _clip01(rouge_n_f1_tokens(pt, gt, 2))
    out["ROUGE-L"] = _clip01(rouge_l_f1_tokens(pt, gt))
    out["ROUGE-SU4"] = _clip01(rouge_su4_f1_tokens(pt, gt, 4))
    out["METEOR"] = _clip01(meteor_tokens(pt, gt))
    out["BLEU-1"] = _clip01(bleu_score_tokens(pt, gt, 1))
    out["BLEU-4"] = _clip01(bleu_score_tokens(pt, gt, 4))
    out["JSD-Sim"] = _clip01(1.0 - jsd_dist_tokens(pt, gt))
    return out

def safe_traditional_metric_row(pred_text, gold_text):
    try:
        return traditional_metric_row(pred_text, gold_text)
    except Exception:
        return {
            "ROUGE-1": np.nan,
            "ROUGE-2": np.nan,
            "ROUGE-L": np.nan,
            "ROUGE-SU4": np.nan,
            "METEOR": np.nan,
            "BLEU-1": np.nan,
            "BLEU-4": np.nan,
            "JSD-Sim": np.nan,
        }

def _nanmean_col(df, col):
    s = pd.to_numeric(df[col], errors="coerce")
    return float(s.mean()) if s.notna().any() else np.nan

def load_metric_store():
    if os.path.exists(SUM_METRIC_JSON):
        with open(SUM_METRIC_JSON, "r") as f:
            return json.load(f)
    return {}

def save_metric_store(store):
    with open(SUM_METRIC_JSON, "w") as f:
        json.dump(store, f, indent=2)

def rebuild_metric_csv_from_store(store):
    rows = []
    for block_name, block in store.items():
        for dataset_name, metrics in block.items():
            row = {"block": block_name, "dataset": dataset_name}
            row.update(metrics)
            rows.append(row)
    out_df = pd.DataFrame(rows)
    out_df.to_csv(SUM_METRIC_CSV, index=False)
    return out_df

def merge_metric_columns(base_df, metric_df, out_csv):
    merged = base_df.copy()
    for c in metric_df.columns:
        merged[c] = metric_df[c].values
    merged.to_csv(out_csv, index=False)
    return merged

def run_block_on_dataset(pred_csv, out_csv, dataset_name, block_name, row_fn, metric_cols):
    df = pd.read_csv(pred_csv).copy()
    for c in ["pred_text", "gold_text"]:
        if c not in df.columns:
            raise KeyError(f"{dataset_name}: missing required column `{c}`")
        df[c] = df[c].map(_normalize_metric_text)

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"{block_name} {dataset_name}", leave=False):
        rows.append(row_fn(r["pred_text"], r["gold_text"]))
    metric_df = pd.DataFrame(rows)

    merged = merge_metric_columns(df, metric_df, out_csv)
    metrics = {c: _nanmean_col(merged, c) for c in metric_cols}
    return metrics, merged

# expose for later cells
_SUMMARY_METRIC_COMMON_READY = True

# -------------------------
# Run lexical/traditional block
# -------------------------
TRADITIONAL_COLS = [
    "ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-SU4",
    "METEOR", "BLEU-1", "BLEU-4", "JSD-Sim"
]

store = load_metric_store()
store.setdefault("lexical_traditional", {})

internal_metrics_a, internal_df_a = run_block_on_dataset(
    SUM_TEST_PRED_CSV,
    SUM_TEST_ENRICHED_CSV,
    "internal_test",
    "lexical_traditional",
    safe_traditional_metric_row,
    TRADITIONAL_COLS
)
store["lexical_traditional"]["internal_test"] = internal_metrics_a

if os.path.exists(SUM_EXT_PRED_CSV):
    external_metrics_a, external_df_a = run_block_on_dataset(
        SUM_EXT_PRED_CSV,
        SUM_EXT_ENRICHED_CSV,
        "external_test",
        "lexical_traditional",
        safe_traditional_metric_row,
        TRADITIONAL_COLS
    )
    store["lexical_traditional"]["external_test"] = external_metrics_a

save_metric_store(store)
summary_df_a = rebuild_metric_csv_from_store(store)

print("LEXICAL / TRADITIONAL RESULTS")
display(summary_df_a[summary_df_a["block"] == "lexical_traditional"])
print("saved:", SUM_TEST_ENRICHED_CSV)
if os.path.exists(SUM_EXT_PRED_CSV):
    print("saved:", SUM_EXT_ENRICHED_CSV)
print("saved:", SUM_METRIC_JSON)
print("saved:", SUM_METRIC_CSV)

In [ ]:
# ============================================================
# DIAGNOSTIC — Why advanced rows are becoming NaN
# ============================================================
dbg_df = pd.read_csv(SUM_TEST_PRED_CSV).copy()

for c in ["pred_text", "gold_text", "source_text"]:
    print(c, "present:", c in dbg_df.columns)
    if c in dbg_df.columns:
        print(c, "null_rate:", float(dbg_df[c].isna().mean()))
        print(c, "sample:", repr(str(dbg_df[c].iloc[0])[:200]))
        print()

dbg_df["pred_text"] = dbg_df["pred_text"].map(_normalize_metric_text)
dbg_df["gold_text"] = dbg_df["gold_text"].map(_normalize_metric_text)
dbg_df["source_text"] = dbg_df["source_text"].map(_normalize_metric_text)

for i in range(min(5, len(dbg_df))):
    r = dbg_df.iloc[i]
    print("=" * 80)
    print("ROW", i)
    print("pred:", repr(r["pred_text"][:150]))
    print("gold:", repr(r["gold_text"][:150]))
    print("source:", repr(r["source_text"][:150]))
    try:
        out = advanced_row(r["pred_text"], r["gold_text"], r["source_text"])
        print("OK")
        print({k: out[k] for k in list(out.keys())[:8]})
    except Exception as e:
        print("FAILED:", type(e).__name__, str(e))
        raise

In [ ]:
# ============================================================
# EXTERNAL RECO-ONLY EVAL USING STAGE-1 RECOMMENDATION CHECKPOINT
# No training. No joint checkpoint. No summarization generation.
# ============================================================

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

PRED_DIR = os.path.join(EVAL_DIR, "stored_predictions")
os.makedirs(PRED_DIR, exist_ok=True)

REC_EXT_STAGE1_PRED_CSV = os.path.join(
    PRED_DIR,
    "external_test_reco_predictions_stage1_rec_only.csv"
)

REC_EXT_STAGE1_METRIC_CSV = os.path.join(
    EVAL_DIR,
    "external_test_reco_metrics_stage1_rec_only.csv"
)

REC_EXT_STAGE1_METRIC_JSON = os.path.join(
    EVAL_DIR,
    "external_test_reco_metrics_stage1_rec_only.json"
)

if not os.path.exists(STAGE1_BEST_CKPT):
    raise FileNotFoundError(
        f"Recommendation-only checkpoint not found: {STAGE1_BEST_CKPT}\n"
        "You need to rerun Stage 1 only. Do not rerun Stage 2/Stage 3 for reco-only external eval."
    )

if ext_loader is None:
    raise RuntimeError(
        "ext_loader is None. External data was not loaded/rebuilt correctly. "
        "Run the setup/data-loader cells first."
    )

print("Loading recommendation-only checkpoint:")
print(STAGE1_BEST_CKPT)

ckpt_obj = torch.load(STAGE1_BEST_CKPT, map_location=device)
print("checkpoint stage:", ckpt_obj.get("stage", "unknown") if isinstance(ckpt_obj, dict) else "raw_state_dict")

load_model_only(STAGE1_BEST_CKPT)
model.eval()

item_proj_table = build_projected_item_table()


def _safe_meta_slice(meta_df, start_idx, bs):
    if meta_df is None:
        return None
    return meta_df.iloc[start_idx:start_idx + bs].reset_index(drop=True)


@torch.no_grad()
def generate_external_reco_predictions_stage1(
    loader,
    out_csv,
    item_proj_table,
    num_negs=REC_EVAL_NEGS,
    seed=12345,
    meta_df=None,
):
    model.eval()
    gen = torch.Generator(device=device).manual_seed(seed)

    rows = []
    global_idx = 0

    for batch in tqdm(loader, desc="external reco predictions with stage1 checkpoint", leave=False):
        batch = to_device(batch)

        h_ids = batch["h_ids"]
        h_act = batch["h_act"]
        h_len = batch["h_len"]
        qid   = batch["qid"]
        bs = h_ids.size(0)

        # CRITICAL: external loader comes from ExtDataset, whose task_id is "sum".
        # For recommendation-only evaluation, force task_id to "rec".
        task_ids = torch.full_like(
            h_len,
            int(TASK2ID["rec"]),
            dtype=torch.long,
            device=device
        )

        hist_item_emb = gather_hist_item_emb(h_ids)
        enc_out = model.encoder.encode_history(hist_item_emb, h_act, h_len, task_ids)

        neg = sample_negs(qid, item_proj_table.size(0), num_negs, gen)
        cand = torch.cat([qid[:, None], neg], dim=1)
        cand_proj = gather_item_proj(item_proj_table, cand)

        logits = model.rec_logits_from_candidates(enc_out["z_hat_q"], cand_proj)
        rank = 1 + (logits[:, 1:] >= logits[:, :1]).sum(dim=1)

        cand_cpu = cand.detach().cpu().numpy()
        logits_cpu = logits.detach().cpu().numpy()
        rank_cpu = rank.detach().cpu().numpy()
        qid_cpu = qid.detach().cpu().numpy()
        hlen_cpu = h_len.detach().cpu().numpy()

        meta_slice = _safe_meta_slice(meta_df, global_idx, bs)

        for i in range(bs):
            order = np.argsort(-logits_cpu[i])
            top_ids = cand_cpu[i][order[:10]].tolist()
            top_scores = logits_cpu[i][order[:10]].astype(float).tolist()

            row = {
                "example_idx": int(global_idx + i),
                "qid": int(qid_cpu[i]),
                "h_len": int(hlen_cpu[i]),
                "rank": int(rank_cpu[i]),
                "pos_score": float(logits_cpu[i][0]),
                "cand_ids_json": json.dumps(cand_cpu[i].astype(int).tolist()),
                "cand_scores_json": json.dumps(logits_cpu[i].astype(float).tolist()),
                "top10_ids_json": json.dumps(top_ids),
                "top10_scores_json": json.dumps(top_scores),
                "checkpoint_used": STAGE1_BEST_CKPT,
                "checkpoint_type": "stage1_rec_only",
                "forced_task_id": "rec",
            }

            if "userid" in batch:
                row["userid"] = str(batch["userid"][i])
            if "sid" in batch:
                row["sid"] = str(batch["sid"][i])
            if "src_nid" in batch:
                row["src_nid"] = int(batch["src_nid"][i].detach().cpu().item())

            if meta_slice is not None:
                for c in meta_slice.columns:
                    if c in {"h_ids", "h_act"}:
                        continue
                    if c not in row:
                        v = meta_slice.loc[i, c]
                        if isinstance(v, np.ndarray):
                            continue
                        row[c] = v

            rows.append(row)

        global_idx += bs

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print("saved external reco predictions:", out_csv)
    print("shape:", df.shape)
    return df


def parse_json_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return json.loads(x)


def auc_from_scores(scores):
    scores = np.asarray(scores, dtype=np.float64)
    pos = scores[0]
    neg = scores[1:]
    if len(neg) == 0:
        return 1.0
    gt = float(np.sum(pos > neg))
    eq = float(np.sum(pos == neg))
    return float((gt + 0.5 * eq) / len(neg))


def swerr_from_arrays(cand_ids, cand_scores, item_emb_table, k=5):
    cand_ids_t = torch.tensor(cand_ids, dtype=torch.long, device=device).unsqueeze(0)
    logits_t = torch.tensor(cand_scores, dtype=torch.float32, device=device).unsqueeze(0)

    order = torch.argsort(logits_t, dim=1, descending=True)
    topk_order = order[:, :min(k, cand_ids_t.size(1))]
    ranked_ids = cand_ids_t.gather(1, topk_order)

    flat_ranked = ranked_ids.reshape(-1)
    ranked_emb = item_emb_table.index_select(0, flat_ranked).view(1, -1, item_emb_table.size(1))

    gold_ids = cand_ids_t[:, 0]
    gold_emb = item_emb_table.index_select(0, gold_ids).unsqueeze(1)

    rel = F.cosine_similarity(
        ranked_emb,
        gold_emb.expand_as(ranked_emb),
        dim=-1
    ).clamp(-1.0, 1.0)

    rel = 0.5 * (rel + 1.0)

    not_sat = torch.ones(1, device=device)
    swerr = torch.zeros(1, device=device)

    for r in range(rel.size(1)):
        rr = rel[:, r].clamp(0.0, 1.0)
        swerr = swerr + not_sat * rr / float(r + 1)
        not_sat = not_sat * (1.0 - rr)

    return float(swerr.item())


def reco_metrics_from_stored_df(df, name):
    item_emb_sem = ITEM_EMB_CPU.to(device)

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"parse reco rows {name}", leave=False):
        rank = int(r["rank"])
        cand_ids = parse_json_list(r["cand_ids_json"])
        cand_scores = parse_json_list(r["cand_scores_json"])

        rows.append({
            "rank": rank,
            "auc": auc_from_scores(cand_scores),
            "swerr5": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=5),
            "swerr10": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=10),
            "swerr20": swerr_from_arrays(cand_ids, cand_scores, item_emb_sem, k=20),
        })

    tmp = pd.DataFrame(rows)
    ranks = tmp["rank"].to_numpy(dtype=np.float64)

    metrics = {
        "MRR": float(np.mean(1.0 / ranks)),
        "Hit@1": float(np.mean(ranks <= 1)),
        "Hit@3": float(np.mean(ranks <= 3)),
        "Hit@5": float(np.mean(ranks <= 5)),
        "Hit@10": float(np.mean(ranks <= 10)),
        "Recall@1": float(np.mean(ranks <= 1)),
        "Recall@3": float(np.mean(ranks <= 3)),
        "Recall@5": float(np.mean(ranks <= 5)),
        "Recall@10": float(np.mean(ranks <= 10)),
        "Precision@1": float(np.mean(ranks <= 1) / 1.0),
        "Precision@3": float(np.mean(ranks <= 3) / 3.0),
        "Precision@5": float(np.mean(ranks <= 5) / 5.0),
        "Precision@10": float(np.mean(ranks <= 10) / 10.0),
        "NDCG@3": float(np.mean((ranks <= 3) * (1.0 / np.log2(ranks + 1.0)))),
        "NDCG@5": float(np.mean((ranks <= 5) * (1.0 / np.log2(ranks + 1.0)))),
        "NDCG@10": float(np.mean((ranks <= 10) * (1.0 / np.log2(ranks + 1.0)))),
        "AUC": float(tmp["auc"].mean()),
        "SWERR@5": float(tmp["swerr5"].mean()),
        "SWERR@10": float(tmp["swerr10"].mean()),
        "SWERR@20": float(tmp["swerr20"].mean()),
        "InvMeanRank": float(1.0 / np.mean(ranks)),
    }

    return metrics


rec_ext_stage1_pred_df = generate_external_reco_predictions_stage1(
    loader=ext_loader,
    out_csv=REC_EXT_STAGE1_PRED_CSV,
    item_proj_table=item_proj_table,
    num_negs=REC_EVAL_NEGS,
    seed=12345,
    meta_df=ext_eval_df.reset_index(drop=True),
)

external_reco_stage1_metrics = reco_metrics_from_stored_df(
    rec_ext_stage1_pred_df,
    "external_test_stage1_rec_only"
)

metric_row = {
    "dataset": "external_test",
    "checkpoint_type": "stage1_rec_only",
    "checkpoint": STAGE1_BEST_CKPT,
    **external_reco_stage1_metrics,
}

metric_df = pd.DataFrame([metric_row])
metric_df.to_csv(REC_EXT_STAGE1_METRIC_CSV, index=False)

with open(REC_EXT_STAGE1_METRIC_JSON, "w") as f:
    json.dump([metric_row], f, indent=2)

print("EXTERNAL RECO METRICS — STAGE1 RECO-ONLY CHECKPOINT")
display(metric_df.T.rename(columns={0: "value"}))

print("saved:", REC_EXT_STAGE1_METRIC_CSV)
print("saved:", REC_EXT_STAGE1_METRIC_JSON)

In [ ]:
# ============================================================
# CELL 17B — Weighted and paraphrased advanced summary metrics
# SELF-CONTAINED FIX
# updates shared json/csv after completion
# ============================================================
import math
import numpy as np
import pandas as pd
from collections import Counter
from functools import lru_cache
from tqdm.auto import tqdm

from nltk.stem import PorterStemmer
from nltk.corpus import wordnet as wn

# -------------------------
# self-contained text helpers
# -------------------------
stemmer_b = PorterStemmer()

@lru_cache(maxsize=500000)
def tok_text_b_cached(text):
    return tuple(_TOKEN_RE.findall(str(text).lower()))

@lru_cache(maxsize=500000)
def tok_text_b_para_cached(text):
    toks = tok_text_b_cached(text)
    out = []
    for t in toks:
        u = stemmer_b.stem(t)
        try:
            syns = wn.synsets(u)
            if len(syns) > 0:
                lemmas = sorted({z.lower().replace("_", " ") for z in syns[0].lemma_names()})
                if len(lemmas) > 0 and " " not in lemmas[0]:
                    u = stemmer_b.stem(lemmas[0])
        except Exception:
            pass
        out.append(u)
    return tuple(out)

def _ngrams_b(tokens, n):
    L = len(tokens)
    if L < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(L - n + 1)]

def rouge_n_f1_b(pred_toks, gold_toks, n=1):
    p = Counter(_ngrams_b(pred_toks, n))
    g = Counter(_ngrams_b(gold_toks, n))
    if len(p) == 0 or len(g) == 0:
        return 0.0
    overlap = sum((p & g).values())
    prec = overlap / max(1, sum(p.values()))
    rec  = overlap / max(1, sum(g.values()))
    return 0.0 if (prec + rec) == 0 else (2.0 * prec * rec / (prec + rec))

def lcs_len_b(a, b):
    n, m = len(a), len(b)
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        ai = a[i - 1]
        for j in range(1, m + 1):
            tmp = dp[j]
            if ai == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]

def rouge_l_f1_b(pred_toks, gold_toks):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    l = lcs_len_b(pred_toks, gold_toks)
    p = l / max(1, len(pred_toks))
    r = l / max(1, len(gold_toks))
    return 0.0 if (p + r) == 0 else (2.0 * p * r / (p + r))

def _skip_bigrams_plus_unigrams_b(tokens, max_skip=4):
    out = []
    L = len(tokens)
    for i in range(L):
        out.append((tokens[i],))
        hi = min(L, i + max_skip + 2)
        for j in range(i + 1, hi):
            out.append((tokens[i], tokens[j]))
    return out

def rouge_su4_f1_b(pred_toks, gold_toks, max_skip=4):
    p = Counter(_skip_bigrams_plus_unigrams_b(pred_toks, max_skip=max_skip))
    g = Counter(_skip_bigrams_plus_unigrams_b(gold_toks, max_skip=max_skip))
    if len(p) == 0 or len(g) == 0:
        return 0.0
    overlap = sum((p & g).values())
    prec = overlap / max(1, sum(p.values()))
    rec  = overlap / max(1, sum(g.values()))
    return 0.0 if (prec + rec) == 0 else (2.0 * prec * rec / (prec + rec))

def meteor_b(pred_toks, gold_toks):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    from nltk.translate.meteor_score import meteor_score
    return float(meteor_score([list(gold_toks)], list(pred_toks)))

def bleu_b(pred_toks, gold_toks, max_n=4):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0

    log_prec_sum = 0.0
    for n in range(1, max_n + 1):
        p = Counter(_ngrams_b(pred_toks, n))
        g = Counter(_ngrams_b(gold_toks, n))
        if len(p) == 0:
            return 0.0
        overlap = sum((p & g).values())
        total = sum(p.values())
        prec_n = (overlap + 1.0) / (total + 1.0)
        log_prec_sum += (1.0 / max_n) * math.log(max(prec_n, 1e-12))

    c = len(pred_toks)
    r = len(gold_toks)
    bp = 1.0 if c > r else math.exp(1.0 - (r / max(c, 1)))
    return float(bp * math.exp(log_prec_sum))

def jsd_sim_b(pred_toks, gold_toks):
    vocab = sorted(set(pred_toks) | set(gold_toks))
    if len(vocab) == 0:
        return 0.0

    pc = Counter(pred_toks)
    gc = Counter(gold_toks)
    p = np.asarray([pc[t] for t in vocab], dtype=np.float64)
    q = np.asarray([gc[t] for t in vocab], dtype=np.float64)

    p = p / max(1e-12, p.sum())
    q = q / max(1e-12, q.sum())
    m = 0.5 * (p + q)

    def _kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / np.maximum(b[mask], 1e-12)))

    dist = float(np.clip(0.5 * _kl(p, m) + 0.5 * _kl(q, m), 0.0, 1.0))
    return _clip01(1.0 - dist)

# -------------------------
# weighted helpers
# -------------------------
def source_token_weight_map_b(source_text):
    toks = list(tok_text_b_cached(source_text))
    ctr = Counter(toks)
    if len(ctr) == 0:
        return {}
    return {k: 1.0 + math.log1p(v) for k, v in ctr.items()}

def token_weight_b(tok, wmap):
    return float(wmap.get(tok, 1.0))

def ngram_weight_b(ng, wmap):
    if len(ng) == 0:
        return 1.0
    return float(np.mean([token_weight_b(t, wmap) for t in ng]))

def weighted_rouge_n_f1_b(pred_toks, gold_toks, source_text, n=1):
    p = Counter(_ngrams_b(pred_toks, n))
    g = Counter(_ngrams_b(gold_toks, n))
    if len(p) == 0 or len(g) == 0:
        return 0.0

    wmap = source_token_weight_map_b(source_text)
    overlap = 0.0
    p_tot = 0.0
    g_tot = 0.0

    for ng, c in p.items():
        p_tot += c * ngram_weight_b(ng, wmap)
    for ng, c in g.items():
        g_tot += c * ngram_weight_b(ng, wmap)
    for ng, c in (p & g).items():
        overlap += c * ngram_weight_b(ng, wmap)

    prec = overlap / max(1e-12, p_tot)
    rec  = overlap / max(1e-12, g_tot)
    return 0.0 if (prec + rec) == 0 else (2.0 * prec * rec / (prec + rec))

def weighted_lcs_sum_b(a, b, wmap):
    n, m = len(a), len(b)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        ai = a[i - 1]
        wai = token_weight_b(ai, wmap)
        for j in range(1, m + 1):
            if ai == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + wai
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

def weighted_rouge_l_f1_b(pred_toks, gold_toks, source_text):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    wmap = source_token_weight_map_b(source_text)
    l = weighted_lcs_sum_b(pred_toks, gold_toks, wmap)
    p_den = sum(token_weight_b(t, wmap) for t in pred_toks)
    r_den = sum(token_weight_b(t, wmap) for t in gold_toks)
    p = l / max(1e-12, p_den)
    r = l / max(1e-12, r_den)
    return 0.0 if (p + r) == 0 else (2.0 * p * r / (p + r))

def weighted_rouge_su4_f1_b(pred_toks, gold_toks, source_text, max_skip=4):
    p = Counter(_skip_bigrams_plus_unigrams_b(pred_toks, max_skip=max_skip))
    g = Counter(_skip_bigrams_plus_unigrams_b(gold_toks, max_skip=max_skip))
    if len(p) == 0 or len(g) == 0:
        return 0.0

    wmap = source_token_weight_map_b(source_text)
    overlap = 0.0
    p_tot = 0.0
    g_tot = 0.0

    for ng, c in p.items():
        p_tot += c * ngram_weight_b(ng, wmap)
    for ng, c in g.items():
        g_tot += c * ngram_weight_b(ng, wmap)
    for ng, c in (p & g).items():
        overlap += c * ngram_weight_b(ng, wmap)

    prec = overlap / max(1e-12, p_tot)
    rec  = overlap / max(1e-12, g_tot)
    return 0.0 if (prec + rec) == 0 else (2.0 * prec * rec / (prec + rec))

def weighted_meteor_b(pred_toks, gold_toks, source_text):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    wmap = source_token_weight_map_b(source_text)
    pc = Counter(pred_toks)
    gc = Counter(gold_toks)

    overlap = 0.0
    p_tot = 0.0
    g_tot = 0.0

    for t, c in pc.items():
        p_tot += c * token_weight_b(t, wmap)
    for t, c in gc.items():
        g_tot += c * token_weight_b(t, wmap)
    for t, c in (pc & gc).items():
        overlap += c * token_weight_b(t, wmap)

    prec = overlap / max(1e-12, p_tot)
    rec  = overlap / max(1e-12, g_tot)
    if prec == 0.0 or rec == 0.0:
        return 0.0
    return float((10.0 * prec * rec) / max(1e-12, rec + 9.0 * prec))

def weighted_bleu_b(pred_toks, gold_toks, source_text, max_n=4):
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    wmap = source_token_weight_map_b(source_text)
    log_prec_sum = 0.0

    for n in range(1, max_n + 1):
        p = Counter(_ngrams_b(pred_toks, n))
        g = Counter(_ngrams_b(gold_toks, n))
        if len(p) == 0:
            return 0.0

        overlap = 0.0
        total = 0.0
        for ng, c in p.items():
            total += c * ngram_weight_b(ng, wmap)
        for ng, c in (p & g).items():
            overlap += c * ngram_weight_b(ng, wmap)

        prec_n = (overlap + 1.0) / (total + 1.0)
        log_prec_sum += (1.0 / max_n) * math.log(max(prec_n, 1e-12))

    c = len(pred_toks)
    r = len(gold_toks)
    bp = 1.0 if c > r else math.exp(1.0 - (r / max(c, 1)))
    return float(bp * math.exp(log_prec_sum))

def weighted_jsd_sim_b(pred_toks, gold_toks, source_text):
    vocab = sorted(set(pred_toks) | set(gold_toks))
    if len(vocab) == 0:
        return 0.0

    wmap = source_token_weight_map_b(source_text)
    pc = Counter(pred_toks)
    gc = Counter(gold_toks)
    p = np.asarray([pc[t] * token_weight_b(t, wmap) for t in vocab], dtype=np.float64)
    q = np.asarray([gc[t] * token_weight_b(t, wmap) for t in vocab], dtype=np.float64)

    p = p / max(1e-12, p.sum())
    q = q / max(1e-12, q.sum())
    m = 0.5 * (p + q)

    def _kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / np.maximum(b[mask], 1e-12)))

    dist = float(np.clip(0.5 * _kl(p, m) + 0.5 * _kl(q, m), 0.0, 1.0))
    return _clip01(1.0 - dist)

# -------------------------
# main row function
# -------------------------
def advanced_row(pred_text, gold_text, source_text):
    pred_text = _normalize_metric_text(pred_text)
    gold_text = _normalize_metric_text(gold_text)
    source_text = _normalize_metric_text(source_text)

    pt = tok_text_b_cached(pred_text)
    gt = tok_text_b_cached(gold_text)
    ppara = tok_text_b_para_cached(pred_text)
    gpara = tok_text_b_para_cached(gold_text)
    src = tok_text_b_cached(source_text)

    out = {}

    # weighted
    out["W-ROUGE-1"] = _clip01(weighted_rouge_n_f1_b(pt, gt, source_text, 1))
    out["W-ROUGE-2"] = _clip01(weighted_rouge_n_f1_b(pt, gt, source_text, 2))
    out["W-ROUGE-L"] = _clip01(weighted_rouge_l_f1_b(pt, gt, source_text))
    out["W-ROUGE-SU4"] = _clip01(weighted_rouge_su4_f1_b(pt, gt, source_text, 4))
    out["W-METEOR"] = _clip01(weighted_meteor_b(pt, gt, source_text))
    out["W-BLEU-1"] = _clip01(weighted_bleu_b(pt, gt, source_text, 1))
    out["W-BLEU-4"] = _clip01(weighted_bleu_b(pt, gt, source_text, 4))
    out["W-JSD-Sim"] = _clip01(weighted_jsd_sim_b(pt, gt, source_text))

    # paraphrase
    out["P-ROUGE-1"] = _clip01(rouge_n_f1_b(ppara, gpara, 1))
    out["P-ROUGE-2"] = _clip01(rouge_n_f1_b(ppara, gpara, 2))
    out["P-ROUGE-L"] = _clip01(rouge_l_f1_b(ppara, gpara))
    out["P-ROUGE-SU4"] = _clip01(rouge_su4_f1_b(ppara, gpara, 4))
    out["P-METEOR"] = _clip01(meteor_b(ppara, gpara))
    out["P-BLEU-1"] = _clip01(bleu_b(ppara, gpara, 1))
    out["P-BLEU-4"] = _clip01(bleu_b(ppara, gpara, 4))
    out["P-JSD-Sim"] = _clip01(jsd_sim_b(ppara, gpara))

    # BFI
    sim_pred_src_rougel = rouge_l_f1_b(pt, src)
    sim_gold_src_rougel = rouge_l_f1_b(gt, src)
    out["BFI-ROUGE-L"] = _clip01(1.0 - abs(sim_pred_src_rougel - sim_gold_src_rougel))

    sim_pred_src_su4 = rouge_su4_f1_b(pt, src, 4)
    sim_gold_src_su4 = rouge_su4_f1_b(gt, src, 4)
    out["BFI-ROUGE-SU4"] = _clip01(1.0 - abs(sim_pred_src_su4 - sim_gold_src_su4))

    sim_pred_src_mtr = meteor_b(pt, src)
    sim_gold_src_mtr = meteor_b(gt, src)
    out["BFI-METEOR"] = _clip01(1.0 - abs(sim_pred_src_mtr - sim_gold_src_mtr))

    sim_pred_src_bleu4 = bleu_b(pt, src, 4)
    sim_gold_src_bleu4 = bleu_b(gt, src, 4)
    out["BFI-BLEU-4"] = _clip01(1.0 - abs(sim_pred_src_bleu4 - sim_gold_src_bleu4))

    sim_pred_src_jsd = jsd_sim_b(pt, src)
    sim_gold_src_jsd = jsd_sim_b(gt, src)
    out["BFI-JSD-Sim"] = _clip01(1.0 - abs(sim_pred_src_jsd - sim_gold_src_jsd))

    return out

def safe_advanced_row(pred_text, gold_text, source_text):
    try:
        return advanced_row(pred_text, gold_text, source_text)
    except Exception:
        return {
            "W-ROUGE-1": np.nan, "W-ROUGE-2": np.nan, "W-ROUGE-L": np.nan, "W-ROUGE-SU4": np.nan,
            "W-METEOR": np.nan, "W-BLEU-1": np.nan, "W-BLEU-4": np.nan, "W-JSD-Sim": np.nan,
            "P-ROUGE-1": np.nan, "P-ROUGE-2": np.nan, "P-ROUGE-L": np.nan, "P-ROUGE-SU4": np.nan,
            "P-METEOR": np.nan, "P-BLEU-1": np.nan, "P-BLEU-4": np.nan, "P-JSD-Sim": np.nan,
            "BFI-ROUGE-L": np.nan, "BFI-ROUGE-SU4": np.nan, "BFI-METEOR": np.nan,
            "BFI-BLEU-4": np.nan, "BFI-JSD-Sim": np.nan,
        }

def run_block_on_dataset_adv(pred_csv, out_csv, dataset_name, block_name):
    df = pd.read_csv(pred_csv).copy()
    for c in ["pred_text", "gold_text", "source_text"]:
        if c not in df.columns:
            raise KeyError(f"{dataset_name}: missing required column `{c}`")
        df[c] = df[c].map(_normalize_metric_text)

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"{block_name} {dataset_name}", leave=False):
        rows.append(safe_advanced_row(r["pred_text"], r["gold_text"], r["source_text"]))
    metric_df = pd.DataFrame(rows)

    merged = df.copy()
    for c in metric_df.columns:
        merged[c] = metric_df[c].values
    merged.to_csv(out_csv, index=False)

    metric_cols = list(metric_df.columns)
    metrics = {c: _nanmean_col(merged, c) for c in metric_cols}
    return metrics, merged

store = load_metric_store()
store.setdefault("weighted_paraphrased_bfi", {})

internal_metrics_b, internal_df_b = run_block_on_dataset_adv(
    SUM_TEST_PRED_CSV,
    SUM_TEST_ENRICHED_CSV,
    "internal_test",
    "weighted_paraphrased_bfi"
)
store["weighted_paraphrased_bfi"]["internal_test"] = internal_metrics_b

if os.path.exists(SUM_EXT_PRED_CSV):
    external_metrics_b, external_df_b = run_block_on_dataset_adv(
        SUM_EXT_PRED_CSV,
        SUM_EXT_ENRICHED_CSV,
        "external_test",
        "weighted_paraphrased_bfi"
    )
    store["weighted_paraphrased_bfi"]["external_test"] = external_metrics_b

save_metric_store(store)
summary_df_b = rebuild_metric_csv_from_store(store)

print("WEIGHTED / PARAPHRASED / BFI RESULTS")
display(summary_df_b[summary_df_b["block"] == "weighted_paraphrased_bfi"])
print("saved:", SUM_TEST_ENRICHED_CSV)
if os.path.exists(SUM_EXT_PRED_CSV):
    print("saved:", SUM_EXT_ENRICHED_CSV)
print("saved:", SUM_METRIC_JSON)
print("saved:", SUM_METRIC_CSV)

In [ ]:
# ============================================================
# CELL 18 — PerSEval setup, feasibility, strict sigma registry
# formula-faithful, no fallback
# uses ONLY the accuracy metric families produced in Cell 17
# ============================================================
import math, json
import numpy as np
import pandas as pd
from functools import lru_cache
from tqdm.auto import tqdm

PSE_ALPHA = 3.0
PSE_BETA  = 1.7
PSE_GAMMA = 4.0
PSE_EPS   = 1e-8

PSE_SIGMA_METRICS = [
    # raw overlap
    "ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-SU4", "METEOR", "BLEU-1", "BLEU-4", "JSD-Sim",
    # weighted
    "W-ROUGE-1", "W-ROUGE-2", "W-ROUGE-L", "W-ROUGE-SU4", "W-METEOR", "W-BLEU-1", "W-BLEU-4", "W-JSD-Sim",
    # paraphrase
    "P-ROUGE-1", "P-ROUGE-2", "P-ROUGE-L", "P-ROUGE-SU4", "P-METEOR", "P-BLEU-1", "P-BLEU-4", "P-JSD-Sim",
    # embedding
    "SBERT-Cos", "BERTScore-F1", "InfoLM-Sim",
    # bounded faithfulness-induced
    "BFI-ROUGE-L", "BFI-ROUGE-SU4", "BFI-METEOR", "BFI-BLEU-4", "BFI-JSD-Sim",
    # "BFI-SBERT-Cos", "BFI-BERTScore-F1", "BFI-InfoLM-Sim",
]

def build_perseval_context_from_enriched(enriched_df):
    df = enriched_df.copy().reset_index(drop=True)

    needed = ["sid", "userid", "qid", "src_nid", "source_text", "gold_text", "pred_text"]
    miss = [c for c in needed if c not in df.columns]
    if miss:
        raise KeyError(f"Enriched summary prediction file missing columns: {miss}")

    df["sid"] = df["sid"].astype(str)
    df["userid"] = df["userid"].astype(str)
    df["qid"] = pd.to_numeric(df["qid"], errors="coerce").fillna(-1).astype(int)
    df["src_nid"] = pd.to_numeric(df["src_nid"], errors="coerce").fillna(-1).astype(int)
    df["doc_key"] = df["src_nid"].astype(str)
    df["user_key"] = df["userid"].astype(str)

    df["gold_text"] = df["gold_text"].astype(str)
    df["pred_text"] = df["pred_text"].astype(str)
    df["doc_text"] = df["source_text"].astype(str)

    df["gold_text_norm"] = (
        df["gold_text"]
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

def triangulation_report(ctx_df, name):
    rep = (
        ctx_df.groupby("doc_key")
        .agg(
            n_rows=("sid", "size"),
            n_users=("user_key", "nunique"),
            n_gold=("gold_text_norm", "nunique"),
            n_sids=("sid", "nunique"),
        )
        .reset_index()
    )
    rep["triangulation_ok"] = ((rep["n_users"] >= 2) & (rep["n_gold"] >= 2)).astype(int)

    print(f"\n===== {name} TRIANGULATION CHECK =====")
    print("docs total:", int(len(rep)))
    print("docs with >=2 users:", int((rep["n_users"] >= 2).sum()))
    print("docs with >=2 distinct gold summaries:", int((rep["n_gold"] >= 2).sum()))
    print("docs satisfying PerSEval triangulation:", int(rep["triangulation_ok"].sum()))

    display(rep.sort_values(["triangulation_ok", "n_users", "n_gold", "n_rows"], ascending=False).head(20))

    if int(rep["triangulation_ok"].sum()) <= 0:
        raise AssertionError(f"{name}: no doc anchor satisfies strict PerSEval triangulation")

    keep_docs = set(rep.loc[rep["triangulation_ok"] == 1, "doc_key"].tolist())
    ctx_keep = ctx_df[ctx_df["doc_key"].isin(keep_docs)].copy().reset_index(drop=True)
    return rep, ctx_keep

def sym_pair_key(a, b):
    a = str(a)
    b = str(b)
    return (a, b) if a <= b else (b, a)

# ----------
# Semantic pair caches needed for semantic sigma metrics
# ----------
@torch.no_grad()
def build_semantic_pair_caches(ctx_df):
    pair_keys = set()

    for _, g in ctx_df.groupby("doc_key", sort=False):
        g = g.reset_index(drop=True)
        preds = g["pred_text"].astype(str).tolist()
        golds = g["gold_text"].astype(str).tolist()
        doc_text = str(g.loc[0, "doc_text"])

        for x in preds:
            pair_keys.add(sym_pair_key(x, doc_text))
        for x in golds:
            pair_keys.add(sym_pair_key(x, doc_text))
        for x in preds:
            for y in preds:
                pair_keys.add(sym_pair_key(x, y))
        for x in golds:
            for y in golds:
                pair_keys.add(sym_pair_key(x, y))
        for x, y in zip(preds, golds):
            pair_keys.add(sym_pair_key(x, y))

    pair_keys = list(pair_keys)
    left = [a for a, b in pair_keys]
    right = [b for a, b in pair_keys]

    print("unique semantic pairs to cache:", len(pair_keys))

    # SBERT pair cache from single text embeddings
    sbert_text_cache = embed_unique_texts_sbert(left + right, batch_size=TEXT_EMB_BATCH)
    sbert_pair_cache = {}
    for a, b in tqdm(pair_keys, desc="SBERT pair cache", leave=False):
        sbert_pair_cache[(a, b)] = sbert_cos_from_cache(a, b, sbert_text_cache)

    # BERTScore pair cache
    bertscore_vals = batched_bertscore_f1(left, right, model_type=BERTSCORE_MODEL_NAME, batch_size=PAIR_METRIC_BATCH)
    bertscore_pair_cache = {(a, b): float(v) for (a, b), v in zip(pair_keys, bertscore_vals)}

    # InfoLM pair cache
    infolm_vals = batched_infolm_sim(left, right, model_name=INFOLM_MODEL_NAME, measure=INFOLM_MEASURE, batch_size=PAIR_METRIC_BATCH)
    infolm_pair_cache = {(a, b): float(v) for (a, b), v in zip(pair_keys, infolm_vals)}

    return {
        "sbert": sbert_pair_cache,
        "bertscore": bertscore_pair_cache,
        "infolm": infolm_pair_cache,
    }

# ----------
# Similarity and sigma
# similarity = higher better
# sigma = 1 - similarity (bounded distance)
# ----------
@lru_cache(maxsize=800000)
def pair_similarity_lex(metric_name, a, b, doc_text):
    a = str(a)
    b = str(b)
    doc_text = str(doc_text)

    ta = tok_text_cached(a)
    tb = tok_text_cached(b)
    pa = tok_text_para_cached(a)
    pb = tok_text_para_cached(b)
    tdoc = tok_text_cached(doc_text)

    if metric_name == "ROUGE-1":
        return _clip01(rouge_n_f1_tokens(ta, tb, 1))
    if metric_name == "ROUGE-2":
        return _clip01(rouge_n_f1_tokens(ta, tb, 2))
    if metric_name == "ROUGE-L":
        return _clip01(rouge_l_f1_tokens(ta, tb))
    if metric_name == "ROUGE-SU4":
        return _clip01(rouge_su4_f1_tokens(ta, tb, 4))
    if metric_name == "METEOR":
        return _clip01(meteor_tokens(ta, tb))
    if metric_name == "BLEU-1":
        return _clip01(bleu_score_tokens(ta, tb, max_n=1, weighted=False))
    if metric_name == "BLEU-4":
        return _clip01(bleu_score_tokens(ta, tb, max_n=4, weighted=False))
    if metric_name == "JSD-Sim":
        return _clip01(1.0 - jsd_dist_tokens(ta, tb, weighted=False))

    if metric_name == "W-ROUGE-1":
        return _clip01(weighted_rouge_n_f1_tokens(ta, tb, doc_text, 1))
    if metric_name == "W-ROUGE-2":
        return _clip01(weighted_rouge_n_f1_tokens(ta, tb, doc_text, 2))
    if metric_name == "W-ROUGE-L":
        return _clip01(weighted_rouge_l_f1_tokens(ta, tb, doc_text))
    if metric_name == "W-ROUGE-SU4":
        return _clip01(weighted_rouge_su4_f1_tokens(ta, tb, doc_text, 4))
    if metric_name == "W-METEOR":
        return _clip01(weighted_meteor_tokens(ta, tb, doc_text))
    if metric_name == "W-BLEU-1":
        return _clip01(bleu_score_tokens(ta, tb, max_n=1, weighted=True, source_text=doc_text))
    if metric_name == "W-BLEU-4":
        return _clip01(bleu_score_tokens(ta, tb, max_n=4, weighted=True, source_text=doc_text))
    if metric_name == "W-JSD-Sim":
        return _clip01(1.0 - jsd_dist_tokens(ta, tb, weighted=True, source_text=doc_text))

    if metric_name == "P-ROUGE-1":
        return _clip01(rouge_n_f1_tokens(pa, pb, 1))
    if metric_name == "P-ROUGE-2":
        return _clip01(rouge_n_f1_tokens(pa, pb, 2))
    if metric_name == "P-ROUGE-L":
        return _clip01(rouge_l_f1_tokens(pa, pb))
    if metric_name == "P-ROUGE-SU4":
        return _clip01(rouge_su4_f1_tokens(pa, pb, 4))
    if metric_name == "P-METEOR":
        return _clip01(meteor_tokens(pa, pb))
    if metric_name == "P-BLEU-1":
        return _clip01(bleu_score_tokens(pa, pb, max_n=1, weighted=False))
    if metric_name == "P-BLEU-4":
        return _clip01(bleu_score_tokens(pa, pb, max_n=4, weighted=False))
    if metric_name == "P-JSD-Sim":
        return _clip01(1.0 - jsd_dist_tokens(pa, pb, weighted=False))

    if metric_name == "BFI-ROUGE-L":
        return _clip01(1.0 - abs(rouge_l_f1_tokens(ta, tdoc) - rouge_l_f1_tokens(tb, tdoc)))
    if metric_name == "BFI-ROUGE-SU4":
        return _clip01(1.0 - abs(rouge_su4_f1_tokens(ta, tdoc, 4) - rouge_su4_f1_tokens(tb, tdoc, 4)))
    if metric_name == "BFI-METEOR":
        return _clip01(1.0 - abs(meteor_tokens(ta, tdoc) - meteor_tokens(tb, tdoc)))
    if metric_name == "BFI-BLEU-4":
        return _clip01(1.0 - abs(bleu_score_tokens(ta, tdoc, max_n=4, weighted=False) - bleu_score_tokens(tb, tdoc, max_n=4, weighted=False)))
    if metric_name == "BFI-JSD-Sim":
        s1 = 1.0 - jsd_dist_tokens(ta, tdoc, weighted=False)
        s2 = 1.0 - jsd_dist_tokens(tb, tdoc, weighted=False)
        return _clip01(1.0 - abs(s1 - s2))

    raise KeyError(metric_name)

def pair_similarity(metric_name, a, b, doc_text, sem_caches):
    a = str(a)
    b = str(b)
    doc_text = str(doc_text)

    if metric_name == "SBERT-Cos":
        return _clip01(sem_caches["sbert"][sym_pair_key(a, b)])
    if metric_name == "BERTScore-F1":
        return _clip01(sem_caches["bertscore"][sym_pair_key(a, b)])
    if metric_name == "InfoLM-Sim":
        return _clip01(sem_caches["infolm"][sym_pair_key(a, b)])

    if metric_name == "BFI-SBERT-Cos":
        return _clip01(1.0 - abs(
            sem_caches["sbert"][sym_pair_key(a, doc_text)] -
            sem_caches["sbert"][sym_pair_key(b, doc_text)]
        ))
    if metric_name == "BFI-BERTScore-F1":
        return _clip01(1.0 - abs(
            sem_caches["bertscore"][sym_pair_key(a, doc_text)] -
            sem_caches["bertscore"][sym_pair_key(b, doc_text)]
        ))
    if metric_name == "BFI-InfoLM-Sim":
        return _clip01(1.0 - abs(
            sem_caches["infolm"][sym_pair_key(a, doc_text)] -
            sem_caches["infolm"][sym_pair_key(b, doc_text)]
        ))

    return pair_similarity_lex(metric_name, a, b, doc_text)

def pair_sigma(metric_name, a, b, doc_text, sem_caches):
    return _clip01(1.0 - pair_similarity(metric_name, a, b, doc_text, sem_caches))

# -------------------------
# Load enriched predictions and prepare contexts
# -------------------------
test_sum_enriched_df = pd.read_csv(SUM_TEST_ENRICHED_CSV)
test_ctx_raw = build_perseval_context_from_enriched(test_sum_enriched_df)
test_tri_report, test_ctx = triangulation_report(test_ctx_raw, "INTERNAL TEST")

if os.path.exists(SUM_EXT_ENRICHED_CSV):
    ext_sum_enriched_df = pd.read_csv(SUM_EXT_ENRICHED_CSV)
    ext_ctx_raw = build_perseval_context_from_enriched(ext_sum_enriched_df)
    ext_tri_report, ext_ctx = triangulation_report(ext_ctx_raw, "EXTERNAL TEST")
else:
    ext_ctx = None
    ext_tri_report = None
    print("external enriched summary csv missing; external PerSEval will skip")

print("building semantic caches for INTERNAL TEST")
test_sem_caches = build_semantic_pair_caches(test_ctx)

if ext_ctx is not None:
    print("building semantic caches for EXTERNAL TEST")
    ext_sem_caches = build_semantic_pair_caches(ext_ctx)
else:
    ext_sem_caches = None

In [ ]:
# ============================================================
# CELL 19 — Strict faithful PerSEval computation and results
# internal + external
# uses ONLY sigma metrics from Cell 17 metric families
# ============================================================
PSE_INTERNAL_SUITE_CSV = os.path.join(EVAL_DIR, "internal_test_perseval_suite.csv")
PSE_INTERNAL_ROWS_CSV  = os.path.join(EVAL_DIR, "internal_test_perseval_rows.csv")
PSE_INTERNAL_DOCS_CSV  = os.path.join(EVAL_DIR, "internal_test_perseval_docs.csv")

PSE_EXTERNAL_SUITE_CSV = os.path.join(EVAL_DIR, "external_test_perseval_suite.csv")
PSE_EXTERNAL_ROWS_CSV  = os.path.join(EVAL_DIR, "external_test_perseval_rows.csv")
PSE_EXTERNAL_DOCS_CSV  = os.path.join(EVAL_DIR, "external_test_perseval_docs.csv")

PSE_JSON = os.path.join(EVAL_DIR, "perseval_results.json")

def compute_perseval_metric(
    ctx_df,
    metric_name,
    sem_caches,
    alpha=PSE_ALPHA,
    beta=PSE_BETA,
    gamma=PSE_GAMMA,
    eps=PSE_EPS,
):
    row_records = []
    doc_records = []

    grouped = ctx_df.groupby("doc_key", sort=False)

    for doc_key, g in tqdm(grouped, desc=f"PerSEval {metric_name}", leave=False):
        g = g.reset_index(drop=True)

        # strict triangulation already filtered in Cell 18, but keep exact assertion
        if g["user_key"].nunique() < 2:
            continue
        if g["gold_text_norm"].nunique() < 2:
            continue

        preds = g["pred_text"].astype(str).tolist()
        golds = g["gold_text"].astype(str).tolist()
        users = g["user_key"].astype(str).tolist()
        doc_text = str(g.loc[0, "doc_text"])

        n = len(g)

        sigma_uu = np.zeros((n, n), dtype=np.float64)
        sigma_ss = np.zeros((n, n), dtype=np.float64)
        sigma_ud = np.zeros((n,), dtype=np.float64)
        sigma_sd = np.zeros((n,), dtype=np.float64)
        sigma_acc = np.zeros((n,), dtype=np.float64)

        for j in range(n):
            sigma_ud[j] = pair_sigma(metric_name, golds[j], doc_text, doc_text, sem_caches)
            sigma_sd[j] = pair_sigma(metric_name, preds[j], doc_text, doc_text, sem_caches)
            sigma_acc[j] = pair_sigma(metric_name, preds[j], golds[j], doc_text, sem_caches)

        for j in range(n):
            for k in range(n):
                sigma_uu[j, k] = pair_sigma(metric_name, golds[j], golds[k], doc_text, sem_caches)
                sigma_ss[j, k] = pair_sigma(metric_name, preds[j], preds[k], doc_text, sem_caches)

        # Faithful DEGRESS
        degress = np.zeros((n,), dtype=np.float64)
        for j in range(n):
            wu = sigma_uu[j] / np.maximum(sigma_ud[j], eps)
            ws = sigma_ss[j] / np.maximum(sigma_sd[j], eps)

            exp_wu = np.exp(wu - np.max(wu))
            exp_ws = np.exp(ws - np.max(ws))

            Xu = (exp_wu / np.maximum(exp_wu.sum(), eps)) * sigma_uu[j]
            Ys = (exp_ws / np.maximum(exp_ws.sum(), eps)) * sigma_ss[j]

            degress[j] = np.mean((np.minimum(Xu, Ys) + eps) / (np.maximum(Xu, Ys) + eps))

        # Faithful ADP / ACP / EDP
        sigma_star = float(np.min(sigma_acc))
        sigma_mean = float(np.mean(sigma_acc))

        ADP = 1.0 / (
            1.0 + (10.0 ** gamma) * np.exp(
                -10.0 * ((sigma_star - 0.0) / ((1.0 - sigma_star) + eps))
            )
        )
        ACP = 1.0 / (
            1.0 + (10.0 ** gamma) * np.exp(
                -10.0 * ((sigma_acc - sigma_star) / ((sigma_mean - sigma_star) + eps))
            )
        )
        DGP = ADP + ACP
        EDP = 1.0 - (
            1.0 / (
                1.0 + (10.0 ** alpha) * np.exp(-(10.0 * beta * DGP))
            )
        )

        PSE = degress * EDP

        doc_records.append({
            "doc_key": str(doc_key),
            "distance_measure": metric_name,
            "n_users": int(g["user_key"].nunique()),
            "n_rows": int(n),
            "ADP": float(ADP),
            "DEGRESS_doc_mean": float(np.mean(degress)),
            "PerSEval_doc_mean": float(np.mean(PSE)),
        })

        for j in range(n):
            row_records.append({
                "doc_key": str(doc_key),
                "sid": str(g.loc[j, "sid"]),
                "user_key": str(users[j]),
                "distance_measure": metric_name,
                "DEGRESS": float(degress[j]),
                "ADP": float(ADP),
                "ACP": float(ACP[j]),
                "DGP": float(DGP[j]),
                "EDP": float(EDP[j]),
                "PerSEval": float(PSE[j]),
                "sigma_acc_dist": float(sigma_acc[j]),
            })

    rows_df = pd.DataFrame(row_records)
    docs_df = pd.DataFrame(doc_records)

    summary_row = {
        "distance_measure": metric_name,
        "alpha": float(alpha),
        "beta": float(beta),
        "gamma": float(gamma),
        "PerSEval_system": float(docs_df["PerSEval_doc_mean"].mean()) if len(docs_df) else float("nan"),
        "DEGRESS_system": float(docs_df["DEGRESS_doc_mean"].mean()) if len(docs_df) else float("nan"),
        "n_docs_used": int(docs_df["doc_key"].nunique()) if len(docs_df) else 0,
        "n_doc_user_pairs_used": int(len(rows_df)),
    }
    return summary_row, rows_df, docs_df

def compute_perseval_suite(ctx_df, sem_caches, suite_name):
    suite_rows = []
    all_rows = []
    all_docs = []

    for m in tqdm(PSE_SIGMA_METRICS, desc=f"PerSEval suite {suite_name}", leave=True):
        summary_row, rows_df, docs_df = compute_perseval_metric(
            ctx_df=ctx_df,
            metric_name=m,
            sem_caches=sem_caches,
            alpha=PSE_ALPHA,
            beta=PSE_BETA,
            gamma=PSE_GAMMA,
            eps=PSE_EPS,
        )
        suite_rows.append(summary_row)
        all_rows.append(rows_df)
        all_docs.append(docs_df)

    suite_df = pd.DataFrame(suite_rows).sort_values("PerSEval_system", ascending=False).reset_index(drop=True)
    rows_df = pd.concat(all_rows, axis=0, ignore_index=True) if len(all_rows) else pd.DataFrame()
    docs_df = pd.concat(all_docs, axis=0, ignore_index=True) if len(all_docs) else pd.DataFrame()
    return suite_df, rows_df, docs_df

# -------------------------
# INTERNAL TEST PerSEval
# -------------------------
internal_pse_suite_df, internal_pse_rows_df, internal_pse_docs_df = compute_perseval_suite(
    ctx_df=test_ctx,
    sem_caches=test_sem_caches,
    suite_name="internal_test",
)

print("INTERNAL TEST PerSEval RESULTS")
display(internal_pse_suite_df)

internal_pse_suite_df.to_csv(PSE_INTERNAL_SUITE_CSV, index=False)
internal_pse_rows_df.to_csv(PSE_INTERNAL_ROWS_CSV, index=False)
internal_pse_docs_df.to_csv(PSE_INTERNAL_DOCS_CSV, index=False)

# -------------------------
# EXTERNAL TEST PerSEval
# -------------------------
if ext_ctx is not None and ext_sem_caches is not None:
    external_pse_suite_df, external_pse_rows_df, external_pse_docs_df = compute_perseval_suite(
        ctx_df=ext_ctx,
        sem_caches=ext_sem_caches,
        suite_name="external_test",
    )

    print("EXTERNAL TEST PerSEval RESULTS")
    display(external_pse_suite_df)

    external_pse_suite_df.to_csv(PSE_EXTERNAL_SUITE_CSV, index=False)
    external_pse_rows_df.to_csv(PSE_EXTERNAL_ROWS_CSV, index=False)
    external_pse_docs_df.to_csv(PSE_EXTERNAL_DOCS_CSV, index=False)
else:
    external_pse_suite_df = None
    print("external PerSEval skipped")

payload = {
    "internal_test": internal_pse_suite_df.to_dict(orient="records"),
    "external_test": None if ext_ctx is None or ext_sem_caches is None else external_pse_suite_df.to_dict(orient="records"),
}
with open(PSE_JSON, "w") as f:
    json.dump(payload, f, indent=2)

print("saved:", PSE_INTERNAL_SUITE_CSV)
print("saved:", PSE_INTERNAL_ROWS_CSV)
print("saved:", PSE_INTERNAL_DOCS_CSV)
if ext_ctx is not None and ext_sem_caches is not None:
    print("saved:", PSE_EXTERNAL_SUITE_CSV)
    print("saved:", PSE_EXTERNAL_ROWS_CSV)
    print("saved:", PSE_EXTERNAL_DOCS_CSV)
print("saved:", PSE_JSON)